## Loading data from AWS using duckDB

In [1]:
import sys
sys.path.insert(0, r'C:\projects\hedge_fund')

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from data.store.duckdb_client import DuckDBClient
from data.store.s3_client import S3Client

db = DuckDBClient()
s  = S3Client()

# Load all clean USDINR data (both contracts)
all_files  = s.list_files('processed/1s/USDINR26SEPFUT/')
all_files += s.list_files('processed/1s/USDINR26OCTFUT/')

all_df = []
for f in all_files:
    parts = f.split('/')
    sym   = parts[2]
    date  = parts[3].replace('.parquet','')
    try:
        df = db.read_parquet(f'processed/1s/{sym}/{date}.parquet')
        df['ist_sec'] = (df['ts_sec'] + 19800) % 86400
        df = df[(df['ist_sec'] >= 9*3600) &
                (df['ist_sec'] <= 17*3600)].reset_index(drop=True)
        df['date']   = date
        df['symbol'] = sym
        all_df.append(df)
        print(f'{sym}/{date}: {len(df)} bars')
    except Exception as e:
        print(f'ERROR {sym}/{date}: {e}')

db.close()
df_raw = pd.concat(all_df, ignore_index=True).reset_index(drop=True)
print(f'\nTotal: {len(df_raw):,} bars across {df_raw["date"].nunique()} days')
print(f'Dates: {sorted(df_raw["date"].unique())}')
print(f'Columns: {len(df_raw.columns)}')

USDINR26SEPFUT/2026-09-22: 12568 bars
USDINR26SEPFUT/2026-09-23: 11031 bars
USDINR26SEPFUT/2026-09-24: 10385 bars


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

USDINR26SEPFUT/2026-09-25: 15469 bars
USDINR26SEPFUT/2026-09-28: 2921 bars
USDINR26OCTFUT/2026-09-30: 16701 bars
USDINR26OCTFUT/2026-10-01: 16554 bars
USDINR26OCTFUT/2026-10-05: 14944 bars
USDINR26OCTFUT/2026-10-06: 18290 bars


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

USDINR26OCTFUT/2026-10-07: 19336 bars


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (13 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

USDINR26OCTFUT/2026-10-19: 17632 bars


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

USDINR26OCTFUT/2026-10-21: 17745 bars

Total: 302,689 bars across 19 days
Dates: ['2026-09-22', '2026-09-23', '2026-09-24', '2026-09-25', '2026-09-28', '2026-09-30', '2026-10-01', '2026-10-05', '2026-10-06', '2026-10-07', '2026-10-08', '2026-10-09', '2026-10-12', '2026-10-13', '2026-10-14', '2026-10-15', '2026-10-16', '2026-10-19', '2026-10-21']
Columns: 55


## Processing the Raw Data 

In [2]:
df = (df_raw.sort_values(["symbol", "date", "ts_sec"])
    .reset_index(drop=True))
print(df.shape)
print(df["date"].nunique())
print(df.head())

(302689, 55)
19
           symbol      ts_sec     open  high      low  close  volume_delta  \
0  USDINR26OCTFUT  1790739000  96.2725  96.3  96.2725   96.3             1   
1  USDINR26OCTFUT  1790739001  96.3000  96.3  96.3000   96.3             0   
2  USDINR26OCTFUT  1790739002  96.3000  96.3  96.3000   96.3             0   
3  USDINR26OCTFUT  1790739003  96.3000  96.3  96.3000   96.3             0   
4  USDINR26OCTFUT  1790739004  96.3000  96.3  96.3000   96.3             0   

   tick_count       vwap      oi  ...  imbalance_ma_30s  imbalance_ma_60s  \
0           2  96.272647  961002  ...          0.696133          0.696133   
1           1  96.272647  961002  ...          0.671351          0.671351   
2           1  96.272647  961002  ...          0.655217          0.655217   
3           1  96.272647  961002  ...          0.654130          0.654130   
4           1  96.272647  961002  ...          0.653632          0.653632   

   spread_zscore  volume_ratio  price_mom_10s  price

## Verifying duplicate timestamps

In [3]:
duplicates = df.duplicated(subset=[
        "symbol",
        "date",
        "ts_sec"],
    keep=False)
print("Duplicate symbol-date-second rows:", duplicates.sum())

Duplicate symbol-date-second rows: 0


## Before regularization marking the original observations

In [4]:
df["is_observed"] = 1
df["had_update"] = 1

## Regularized each trading day to 1-second time clock 

In [5]:
def regularize_day(day):
    day = day.sort_values("ts_sec").copy()
    start = int(day["ts_sec"].min())
    end = int(day["ts_sec"].max())
    full_seconds = pd.DataFrame({
        "ts_sec": np.arange(
            start,
            end + 1)
    })
    symbol = day["symbol"].iloc[0]
    date = day["date"].iloc[0]
    full_seconds["symbol"] = symbol
    full_seconds["date"] = date

    out = full_seconds.merge(day,
            on=[
            "symbol",
            "date",
            "ts_sec"], how="left")

    return out

In [6]:
df_1s = (
    df
    .groupby(
        ["symbol", "date"],
        group_keys=False
    )
    .apply(regularize_day)
    .reset_index(drop=True)
)

## Fixing is_observed/had_update
### inserted seconds becomes is_observed = 0 and had_update = 0

In [7]:
df_1s["is_observed"] = (
    df_1s["is_observed"]
    .fillna(0)
    .astype(int)
)

df_1s["had_update"] = (
    df_1s["had_update"]
    .fillna(0)
    .astype(int)
)

## Forward Fill Market State
#### Things like:
#### - bid prices
#### - ask prices
#### - bid quantities
#### - ask quantities
#### - open interest
#### are state variables. If nothing happened for three seconds, the most recent state is still the current state.
#### So forward-fill them within each trading day only.

In [8]:
state_cols = [
    "oi",

    "bid_p1", "bid_q1",
    "bid_p2", "bid_q2",
    "bid_p3", "bid_q3",
    "bid_p4", "bid_q4",
    "bid_p5", "bid_q5",

    "ask_p1", "ask_q1",
    "ask_p2", "ask_q2",
    "ask_p3", "ask_q3",
    "ask_p4", "ask_q4",
    "ask_p5", "ask_q5",
]

In [9]:
state_cols = [
    c for c in state_cols
    if c in df_1s.columns
]

In [10]:
df_1s[state_cols] = (
    df_1s
    .groupby(
        ["symbol", "date"]
    )[state_cols]
    .ffill()
)

### Flow variables must not be forward filled 

In [11]:
flow_cols = [
    "volume_delta",
    "tick_count"
]

for col in flow_cols:

    if col in df_1s.columns:

        df_1s[col] = (
            df_1s[col]
            .fillna(0)
        )

### Do not trust old derived features 
#### Some of these were calculated on the irregular update-time data. Once we convert to a proper 1-second clock, we should recompute the features ourselves.So in the clean notebook we would treat raw derived columns as:
####                                        [validation/reference only] not model inputs.
#### Your actual features should come from:
####                                        [bid, ask, depth, volume, tick count, OI] after regularization.

### Recomputing the basic market state 

#### Mid-Price

In [12]:
df_1s["mid"] = (
    df_1s["bid_p1"]
    +
    df_1s["ask_p1"]
) / 2

#### Spread

In [13]:
df_1s["spread"] = (
    df_1s["ask_p1"]
    -
    df_1s["bid_p1"]
)

#### spread_bps = 10^4x(S_t/M_t)

In [14]:
eps = 1e-12

df_1s["spread_bps_calc"] = (
    1e4
    *
    df_1s["spread"]
    /
    (df_1s["mid"] + eps)
)

#### Log-Mid Price 

In [15]:
df_1s["log_mid"] = np.log(
    df_1s["mid"]
)

#### One-second return:    r_t = log(M_t) - log(M_t-1)

In [16]:
df_1s["ret_1s"] = (
    df_1s["log_mid"]
    -
    df_1s
    .groupby(
        ["symbol", "date"]
    )["log_mid"]
    .shift(1)
)

### Recomputing L5 Depth 

In [17]:
bid_cols = [
    f"bid_q{i}"
    for i in range(1, 6)
]

ask_cols = [
    f"ask_q{i}"
    for i in range(1, 6)
]

df_1s["bid_depth_l5"] = (
    df_1s[bid_cols]
    .sum(axis=1)
)

df_1s["ask_depth_l5"] = (
    df_1s[ask_cols]
    .sum(axis=1)
)

df_1s["total_depth_l5"] = (
    df_1s["bid_depth_l5"]
    +
    df_1s["ask_depth_l5"]
)

### Flaging incomplete trading days 

In [18]:
session_summary = (
    df_1s
    .groupby(["symbol", "date"])
    .agg(
        first_sec=("ts_sec", "min"),
        last_sec=("ts_sec", "max"),
        rows=("ts_sec", "size"),
        observed_rows=("is_observed", "sum")
    )
)

session_summary["observation_rate"] = (
    session_summary["observed_rows"]
    /
    session_summary["rows"]
)

print(session_summary)

                            first_sec    last_sec   rows  observed_rows  \
symbol         date                                                       
USDINR26OCTFUT 2026-09-30  1790739000  1790767748  28749          16701   
               2026-10-01  1790825780  1790854181  28402          16554   
               2026-10-05  1791171000  1791199766  28767          14944   
               2026-10-06  1791257400  1791286154  28755          18290   
               2026-10-07  1791343800  1791372551  28752          19336   
               2026-10-08  1791430200  1791458948  28749          20968   
               2026-10-09  1791516600  1791545347  28748          20206   
               2026-10-12  1791775800  1791804553  28754          18406   
               2026-10-13  1791862200  1791890992  28793          18435   
               2026-10-14  1791948600  1791977391  28792          17290   
               2026-10-15  1792035000  1792063796  28797          17281   
               2026-10-16

### Basic sanity check 

In [19]:
print(
    df_1s[
        [
            "mid",
            "spread_bps_calc",
            "ret_1s",
            "volume_delta",
            "tick_count",
            "had_update"
        ]
    ].describe()
)

print(
    "Rows:",
    len(df_1s)
)

print(
    "Dates:",
    df_1s["date"].nunique()
)

print(
    "Observed rate:",
    df_1s["is_observed"].mean()
)

print(
    "1s zero-return rate:",
    (df_1s["ret_1s"] == 0).mean()
)

print(
    "Missing mid:",
    df_1s["mid"].isna().mean()
)

print(
    "Crossed book:",
    (
        df_1s["bid_p1"]
        > df_1s["ask_p1"]
    ).sum()
)

print(
    "Locked book:",
    (
        df_1s["bid_p1"]
        == df_1s["ask_p1"]
    ).sum()
)

                 mid  spread_bps_calc        ret_1s   volume_delta  \
count  524325.000000    524325.000000  5.243060e+05  524325.000000   
mean       95.899992         1.010562 -5.150879e-09       3.781635   
std         0.696022         0.683096  1.712575e-05      98.972485   
min        94.555000         0.257357 -2.207822e-03       0.000000   
25%        95.185000         0.521186  0.000000e+00       0.000000   
50%        95.895000         0.790025  0.000000e+00       0.000000   
75%        96.501250         1.305500  0.000000e+00       0.000000   
max        97.152500        41.844810  2.120478e-03   12408.000000   

          tick_count     had_update  
count  524325.000000  524325.000000  
mean        0.646525       0.577293  
std         0.608537       0.493990  
min         0.000000       0.000000  
25%         0.000000       0.000000  
50%         1.000000       1.000000  
75%         1.000000       1.000000  
max         4.000000       1.000000  
Rows: 524325
Dates: 19
Obse

In [20]:
dt = (
    df_1s
    .groupby(
        ["symbol", "date"]
    )["ts_sec"]
    .diff()
)

print(
    dt.value_counts()
    .sort_index()
    .head(10)
)

ts_sec
1.0    524306
Name: count, dtype: int64


In [21]:
print("\n===== VOLUME DELTA CHECK =====")

print(
    df[df["volume_delta"].notna()]["volume_delta"].describe()
)

print(
    "\nNegative:",
    (df["volume_delta"] < 0).sum()
)

print(
    "Zero:",
    (df["volume_delta"] == 0).sum()
)

print(
    "Positive:",
    (df["volume_delta"] > 0).sum()
)

print(
    "\nSmallest values:"
)

print(
    df.loc[
        df["volume_delta"].notna(),
        [
            "symbol",
            "date",
            "ts_sec",
            "volume_delta",
            "tick_count"
        ]
    ]
    .sort_values("volume_delta")
    .head(20)
)

print(
    "\nLargest values:"
)

print(
    df.loc[
        df["volume_delta"].notna(),
        [
            "symbol",
            "date",
            "ts_sec",
            "volume_delta",
            "tick_count"
        ]
    ]
    .sort_values(
        "volume_delta",
        ascending=False
    )
    .head(20)
)


===== VOLUME DELTA CHECK =====
count    302689.000000
mean          6.550638
std         130.192057
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max       12408.000000
Name: volume_delta, dtype: float64

Negative: 0
Zero: 295504
Positive: 7185

Smallest values:
                symbol        date      ts_sec  volume_delta  tick_count
302687  USDINR26SEPFUT  2026-09-28  1790578793             0           1
1       USDINR26OCTFUT  2026-09-30  1790739001             0           1
2       USDINR26OCTFUT  2026-09-30  1790739002             0           1
3       USDINR26OCTFUT  2026-09-30  1790739003             0           1
4       USDINR26OCTFUT  2026-09-30  1790739004             0           1
5       USDINR26OCTFUT  2026-09-30  1790739005             0           1
6       USDINR26OCTFUT  2026-09-30  1790739006             0           1
7       USDINR26OCTFUT  2026-09-30  1790739007             0           1
8       USDINR26OCTFUT  2026-09-3

In [22]:
import numpy as np
import pandas as pd

print("\n===== VOLUME_DELTA SEMANTIC DIAGNOSTIC =====")

# ---------------------------------------------------------
# 1. Basic proportions
# ---------------------------------------------------------

n = len(df)

print("\nTotal rows:", n)

print(
    "volume_delta > 0:",
    (df["volume_delta"] > 0).sum(),
    "fraction:",
    (df["volume_delta"] > 0).mean()
)

print(
    "tick_count > 0:",
    (df["tick_count"] > 0).sum(),
    "fraction:",
    (df["tick_count"] > 0).mean()
)

print(
    "volume_delta > 0 AND tick_count > 0:",
    (
        (df["volume_delta"] > 0)
        &
        (df["tick_count"] > 0)
    ).sum()
)

print(
    "volume_delta > 0 AND tick_count == 0:",
    (
        (df["volume_delta"] > 0)
        &
        (df["tick_count"] == 0)
    ).sum()
)

# ---------------------------------------------------------
# 2. Volume delta conditional on tick count
# ---------------------------------------------------------

print("\n===== VOLUME DELTA BY TICK COUNT =====")

print(
    df.groupby("tick_count")["volume_delta"]
    .agg(
        count="count",
        positive=lambda x: (x > 0).sum(),
        mean="mean",
        median="median",
        max="max"
    )
)

# ---------------------------------------------------------
# 3. Compare volume events with price changes
# ---------------------------------------------------------

df_tmp = df.sort_values(
    ["symbol", "date", "ts_sec"]
).copy()

mid = (
    df_tmp["bid_p1"]
    +
    df_tmp["ask_p1"]
) / 2.0

df_tmp["_mid"] = mid

df_tmp["_mid_change"] = (
    df_tmp["_mid"]
    -
    df_tmp.groupby(
        ["symbol", "date"]
    )["_mid"].shift(1)
)

volume_event = (
    df_tmp["volume_delta"] > 0
)

price_event = (
    df_tmp["_mid_change"] != 0
)

print("\n===== PRICE CHANGE CONDITIONAL ON VOLUME EVENT =====")

print(
    "P(price move | volume_delta > 0):",
    price_event[volume_event].mean()
)

print(
    "P(price move | volume_delta == 0):",
    price_event[~volume_event].mean()
)

# ---------------------------------------------------------
# 4. Distribution of positive volume values only
# ---------------------------------------------------------

positive_volume = (
    df.loc[
        df["volume_delta"] > 0,
        "volume_delta"
    ]
)

print("\n===== POSITIVE VOLUME DELTA ONLY =====")

print(
    positive_volume.describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

# ---------------------------------------------------------
# 5. Most common positive values
# ---------------------------------------------------------

print("\n===== MOST COMMON POSITIVE VALUES =====")

print(
    positive_volume
    .value_counts()
    .head(30)
)

# ---------------------------------------------------------
# 6. Check whether positive volume is concentrated
# around specific tick counts
# ---------------------------------------------------------

print("\n===== POSITIVE-VOLUME ROWS BY TICK COUNT =====")

print(
    df.loc[
        df["volume_delta"] > 0,
        "tick_count"
    ]
    .value_counts(
        normalize=False
    )
    .sort_index()
)

print("\n===== FRACTION POSITIVE VOLUME WITHIN EACH TICK COUNT =====")

print(
    df.groupby("tick_count")["volume_delta"]
    .apply(
        lambda x: (x > 0).mean()
    )
)


===== VOLUME_DELTA SEMANTIC DIAGNOSTIC =====

Total rows: 302689
volume_delta > 0: 7185 fraction: 0.02373723524805989
tick_count > 0: 302689 fraction: 1.0
volume_delta > 0 AND tick_count > 0: 7185
volume_delta > 0 AND tick_count == 0: 0

===== VOLUME DELTA BY TICK COUNT =====
             count  positive        mean  median    max
tick_count                                             
1           267259         0    0.000000     0.0      0
2            34561      6742   53.855299     0.0  12408
3              868       443  139.991935     1.0  10000
4                1         0    0.000000     0.0      0

===== PRICE CHANGE CONDITIONAL ON VOLUME EVENT =====
P(price move | volume_delta > 0): 0.45288796102992346
P(price move | volume_delta == 0): 0.1163436027938708

===== POSITIVE VOLUME DELTA ONLY =====
count     7185.000000
mean       275.964649
std        799.877956
min          1.000000
1%           1.000000
5%           1.000000
10%          2.000000
25%         11.000000
50%     

### Calculating Basic Quantities 
#### 1. Mid Price 
#### 2. log price
#### 3. return (log returns to be specific)

In [23]:
eps = 1e-12
# ---------------------------------------------------------
# MID PRICE
#---------------------------------------------------------
df_1s["mid"] = (df_1s["bid_p1"]+ df_1s["ask_p1"]) / 2.0
df_1s["log_mid"] = np.log(df_1s["mid"])
g = df_1s.groupby(["symbol", "date"],group_keys=False)
# ---------------------------------------------------------
# 1-SECOND LOG RETURN
# ---------------------------------------------------------
df_1s["ret_1s"] = (df_1s["log_mid"]- g["log_mid"].shift(1))

### OFI Calculation 

In [24]:
# =========================================================
# L1 OFI
# =========================================================

bid_p_prev = g["bid_p1"].shift(1)
bid_q_prev = g["bid_q1"].shift(1)

ask_p_prev = g["ask_p1"].shift(1)
ask_q_prev = g["ask_q1"].shift(1)

bid_of = (
    (df_1s["bid_p1"] >= bid_p_prev).astype(float)
    * df_1s["bid_q1"]
    -
    (df_1s["bid_p1"] <= bid_p_prev).astype(float)
    * bid_q_prev
)

ask_of = (
    -
    (df_1s["ask_p1"] <= ask_p_prev).astype(float)
    * df_1s["ask_q1"]
    +
    (df_1s["ask_p1"] >= ask_p_prev).astype(float)
    * ask_q_prev
)

df_1s["ofi_l1"] = (
    bid_of + ask_of
)

### Signed log transform 

In [25]:
def signed_log(x):
    return np.sign(x) * np.log1p(np.abs(x))

### OFI level and accelaration 

In [26]:
for h in [30, 60, 300]:

    raw_col = f"_ofi_sum_{h}s"

    df_1s[raw_col] = (
        df_1s
        .groupby(["symbol", "date"])["ofi_l1"]
        .transform(
            lambda x: x.rolling(
                h,
                min_periods=max(1, h // 3)
            ).sum()
        )
    )

In [27]:
df_1s["ofi_l1_sum_60s_signed_log"] = (
    signed_log(
        df_1s["_ofi_sum_60s"]
    )
)

In [28]:
ofi_rate_diff = (
    df_1s["_ofi_sum_30s"] / 30.0
    -
    df_1s["_ofi_sum_300s"] / 300.0
)

df_1s["ofi_rate_diff_30_300_sl"] = (
    signed_log(ofi_rate_diff)
)

### Active State 

In [29]:
df_1s["_ofi_active"] = (
    df_1s["ofi_l1"].fillna(0) != 0
).astype(float)

for h in [60, 300]:

    df_1s[f"ofi_activity_{h}s"] = (
        df_1s
        .groupby(["symbol", "date"])["_ofi_active"]
        .transform(
            lambda x: x.rolling(
                h,
                min_periods=max(1, h // 3)
            ).mean()
        )
    )

In [30]:
df_1s["update_rate_300s"] = (
    df_1s
    .groupby(["symbol", "date"])["had_update"]
    .transform(
        lambda x: x.rolling(
            300,
            min_periods=100
        ).mean()
    )
)

In [31]:
df_1s["_price_moved"] = (
    df_1s["ret_1s"].fillna(0) != 0
).astype(float)

df_1s["price_move_rate_30s"] = (
    df_1s
    .groupby(["symbol", "date"])["_price_moved"]
    .transform(
        lambda x: x.rolling(
            30,
            min_periods=10
        ).mean()
    )
)

In [32]:
df_1s["_ret_30s"] = (
    df_1s["log_mid"]
    - g["log_mid"].shift(30)
)

df_1s["_ofi_30_sl"] = (
    signed_log(
        df_1s["_ofi_sum_30s"]
    )
)

In [33]:
impact_window = 1800
impact_min = 600

def build_impact_features(day):

    x = day["_ofi_30_sl"]
    y = day["_ret_30s"]

    mx = x.rolling(
        impact_window,
        min_periods=impact_min
    ).mean()

    my = y.rolling(
        impact_window,
        min_periods=impact_min
    ).mean()

    mxy = (x * y).rolling(
        impact_window,
        min_periods=impact_min
    ).mean()

    mx2 = (x * x).rolling(
        impact_window,
        min_periods=impact_min
    ).mean()

    my2 = (y * y).rolling(
        impact_window,
        min_periods=impact_min
    ).mean()

    cov_xy = mxy - mx * my
    var_x = mx2 - mx * mx
    var_y = my2 - my * my

    lam = cov_xy / (var_x + eps)

    alpha = (
        my
        - lam * mx
    )

    r2 = (
        cov_xy ** 2
        /
        (
            (var_x + eps)
            * (var_y + eps)
        )
    )

    # Parameters used at t must come from t-1
    day["impact_lambda_1800s"] = (
        lam.shift(1)
    )

    day["_impact_alpha_1800s"] = (
        alpha.shift(1)
    )

    day["impact_r2_1800s"] = (
        r2.shift(1)
    )

    predicted = (
        day["_impact_alpha_1800s"]
        +
        day["impact_lambda_1800s"]
        * x
    )

    resid = (
        y - predicted
    )

    # use historical residual volatility
    resid_sigma = (
        resid.shift(1)
        .rolling(
            impact_window,
            min_periods=impact_min
        )
        .std()
    )

    day["ofi_response_gap_z"] = (
        -np.sign(x)
        * resid
        /
        (resid_sigma + eps)
    )

    return day


df_1s = (
    df_1s
    .groupby(
        ["symbol", "date"],
        group_keys=False
    )
    .apply(build_impact_features)
)

### Queue and Depth imbalance 

In [34]:
df_1s["queue_imbalance_l1"] = (
    (
        df_1s["bid_q1"]
        -
        df_1s["ask_q1"]
    )
    /
    (
        df_1s["bid_q1"]
        +
        df_1s["ask_q1"]
        +
        eps
    )
)

In [35]:
bid_depth_cols = [
    f"bid_q{i}"
    for i in range(1, 6)
]

ask_depth_cols = [
    f"ask_q{i}"
    for i in range(1, 6)
]

df_1s["_bid_depth_l5"] = (
    df_1s[bid_depth_cols].sum(axis=1)
)

df_1s["_ask_depth_l5"] = (
    df_1s[ask_depth_cols].sum(axis=1)
)

df_1s["depth_imbalance_l5"] = (
    (
        df_1s["_bid_depth_l5"]
        -
        df_1s["_ask_depth_l5"]
    )
    /
    (
        df_1s["_bid_depth_l5"]
        +
        df_1s["_ask_depth_l5"]
        +
        eps
    )
)

In [36]:
bid_same = (
    df_1s["bid_p1"]
    == bid_p_prev
)

ask_same = (
    df_1s["ask_p1"]
    == ask_p_prev
)

bid_replenish = np.where(
    bid_same,
    np.maximum(
        df_1s["bid_q1"]
        - bid_q_prev,
        0
    )
    /
    (bid_q_prev + eps),
    0.0
)

ask_replenish = np.where(
    ask_same,
    np.maximum(
        df_1s["ask_q1"]
        - ask_q_prev,
        0
    )
    /
    (ask_q_prev + eps),
    0.0
)

df_1s["_queue_replenishment_pressure"] = (
    np.log1p(bid_replenish)
    -
    np.log1p(ask_replenish)
)

df_1s["queue_replenishment_pressure_60s"] = (
    df_1s
    .groupby(["symbol", "date"])[
        "_queue_replenishment_pressure"
    ]
    .transform(
        lambda x: x.rolling(
            60,
            min_periods=20
        ).sum()
    )
)

### Imbalance changes 

In [37]:
df_1s["queue_imbalance_delta"] = (
    df_1s["queue_imbalance_l1"]
    -
    df_1s.groupby(
        ["symbol", "date"]
    )["queue_imbalance_l1"].shift(1)
)

df_1s["depth_imbalance_delta"] = (
    df_1s["depth_imbalance_l5"]
    -
    df_1s.groupby(
        ["symbol", "date"]
    )["depth_imbalance_l5"].shift(1)
)

### Relative spread regimes 

In [38]:
df_1s["_spread_bps"] = (
    1e4
    *
    (
        df_1s["ask_p1"]
        -
        df_1s["bid_p1"]
    )
    /
    (df_1s["mid"] + eps)
)

In [39]:
df_1s["spread_rank_1800s"] = (
    df_1s
    .groupby(["symbol", "date"])["_spread_bps"]
    .transform(
        lambda x: x.rolling(
            1800,
            min_periods=300
        ).rank(
            pct=True
        )
    )
)

### Executed volume delta 

In [40]:
df_1s["_volume_delta_sum_300s"] = (
    df_1s
    .groupby(["symbol", "date"])["volume_delta"]
    .transform(
        lambda x: x.rolling(
            300,
            min_periods=100
        ).sum()
    )
)

df_1s["volume_delta_sum_300s_sl"] = (
    signed_log(
        df_1s["_volume_delta_sum_300s"]
    )
)

### Price displacement / momentum

In [41]:
for h in [10, 30, 60]:

    df_1s[f"price_mom_{h}s_new"] = (
        df_1s["log_mid"]
        -
        df_1s
        .groupby(["symbol", "date"])["log_mid"]
        .shift(h)
    )

### Realized volatility

In [42]:
df_1s["_ret_sq"] = (
    df_1s["ret_1s"] ** 2
)

for h in [60, 300]:

    df_1s[f"_rv_{h}s"] = (
        df_1s
        .groupby(["symbol", "date"])["_ret_sq"]
        .transform(
            lambda x: np.sqrt(
                x.rolling(
                    h,
                    min_periods=max(
                        2,
                        h // 3
                    )
                ).sum()
            )
        )
    )

In [43]:
df_1s["log_rv_300s_new"] = (
    np.log1p(
        df_1s["_rv_300s"]
    )
)

In [44]:
df_1s["rv_ratio_60_300_new"] = (
    df_1s["_rv_60s"]
    /
    (
        df_1s["_rv_300s"]
        + eps
    )
)

### Lag-1 return ACF

In [45]:
df_1s["_ret_lag1"] = (
    df_1s
    .groupby(["symbol", "date"])["ret_1s"]
    .shift(1)
)

df_1s["ret_acf_lag1_300s"] = (
    df_1s
    .groupby(
        ["symbol", "date"],
        group_keys=False
    )
    .apply(
        lambda day:
            day["ret_1s"]
            .rolling(
                300,
                min_periods=120
            )
            .corr(
                day["_ret_lag1"]
            )
    )
)

### Variance Ratio

In [46]:
for q in [5, 30]:

    df_1s[f"_ret_sum_{q}"] = (
        df_1s
        .groupby(["symbol", "date"])["ret_1s"]
        .transform(
            lambda x: x.rolling(
                q,
                min_periods=q
            ).sum()
        )
    )

    var_1 = (
        df_1s
        .groupby(["symbol", "date"])["ret_1s"]
        .transform(
            lambda x: x.rolling(
                300,
                min_periods=120
            ).var()
        )
    )

    var_q = (
        df_1s
        .groupby(["symbol", "date"])[
            f"_ret_sum_{q}"
        ]
        .transform(
            lambda x: x.rolling(
                300,
                min_periods=120
            ).var()
        )
    )

    vr = (
        var_q
        /
        (
            q * var_1
            + eps
        )
    )

    # Log compression.
    # No global quantile clipping here -> avoids future leakage.
    df_1s[f"log_vr_{q}_300s"] = (
        np.log(
            np.maximum(
                vr,
                1e-6
            )
        )
    )

In [47]:
df_1s["_ret_sq_pos"] = np.where(
    df_1s["ret_1s"] > 0,
    df_1s["ret_1s"] ** 2,
    0.0
)

df_1s["_ret_sq_neg"] = np.where(
    df_1s["ret_1s"] < 0,
    df_1s["ret_1s"] ** 2,
    0.0
)

for w in [300, 900]:

    up = (
        df_1s
        .groupby(["symbol", "date"])["_ret_sq_pos"]
        .transform(
            lambda x: x.rolling(
                w,
                min_periods=max(
                    120,
                    w // 3
                )
            ).sum()
        )
    )

    down = (
        df_1s
        .groupby(["symbol", "date"])["_ret_sq_neg"]
        .transform(
            lambda x: x.rolling(
                w,
                min_periods=max(
                    120,
                    w // 3
                )
            ).sum()
        )
    )

    df_1s[
        f"semivar_imbalance_{w}s"
    ] = (
        (up - down)
        /
        (
            up + down
            + eps
        )
    )

### Directional move bias 

In [48]:
df_1s["_up_move"] = (
    df_1s["ret_1s"] > 0
).astype(float)

df_1s["_down_move"] = (
    df_1s["ret_1s"] < 0
).astype(float)

up_count = (
    df_1s
    .groupby(["symbol", "date"])["_up_move"]
    .transform(
        lambda x: x.rolling(
            300,
            min_periods=120
        ).sum()
    )
)

down_count = (
    df_1s
    .groupby(["symbol", "date"])["_down_move"]
    .transform(
        lambda x: x.rolling(
            300,
            min_periods=120
        ).sum()
    )
)

move_count = (
    up_count + down_count
)

df_1s["direction_bias_300s"] = (
    (up_count - down_count)
    /
    (
        move_count
        + eps
    )
)

# Require at least 10 actual nonzero moves
df_1s.loc[
    move_count < 10,
    "direction_bias_300s"
] = np.nan

### Sign-persistence

In [49]:
df_1s["_move_sign"] = np.nan

df_1s.loc[
    df_1s["ret_1s"] > 0,
    "_move_sign"
] = 1.0

df_1s.loc[
    df_1s["ret_1s"] < 0,
    "_move_sign"
] = -1.0


def add_move_transition_features(day):

    move_sign = day["_move_sign"]

    # Most recent nonzero move sign
    previous_nonzero_sign = (
        move_sign
        .ffill()
        .shift(1)
    )

    valid_transition = (
        move_sign.notna()
        &
        previous_nonzero_sign.notna()
    )

    same_transition = (
        valid_transition
        &
        (
            move_sign
            == previous_nonzero_sign
        )
    )

    day["_valid_move_transition"] = (
        valid_transition.astype(float)
    )

    day["_same_move_transition"] = (
        same_transition.astype(float)
    )

    return day


df_1s = (
    df_1s
    .groupby(
        ["symbol", "date"],
        group_keys=False
    )
    .apply(
        add_move_transition_features
    )
)


transition_count = (
    df_1s
    .groupby(
        ["symbol", "date"]
    )["_valid_move_transition"]
    .transform(
        lambda x: x.rolling(
            900,
            min_periods=300
        ).sum()
    )
)

same_count = (
    df_1s
    .groupby(
        ["symbol", "date"]
    )["_same_move_transition"]
    .transform(
        lambda x: x.rolling(
            900,
            min_periods=300
        ).sum()
    )
)

df_1s["sign_persistence_900s"] = (
    same_count
    /
    (
        transition_count
        + eps
    )
)

df_1s.loc[
    transition_count < 20,
    "sign_persistence_900s"
] = np.nan

### Jump Ratio

In [50]:
df_1s["_abs_ret"] = (
    np.abs(
        df_1s["ret_1s"]
    )
)

df_1s["_abs_ret_lag1"] = (
    df_1s
    .groupby(
        ["symbol", "date"]
    )["_abs_ret"]
    .shift(1)
)

df_1s["_bipower_component"] = (
    df_1s["_abs_ret"]
    *
    df_1s["_abs_ret_lag1"]
)

jump_rv = (
    df_1s
    .groupby(["symbol", "date"])["_ret_sq"]
    .transform(
        lambda x: x.rolling(
            300,
            min_periods=120
        ).sum()
    )
)

jump_bv = (
    np.pi / 2.0
    *
    df_1s
    .groupby(
        ["symbol", "date"]
    )["_bipower_component"]
    .transform(
        lambda x: x.rolling(
            300,
            min_periods=120
        ).sum()
    )
)

jump_var = (
    jump_rv - jump_bv
).clip(
    lower=0
)

df_1s["jump_ratio_300s"] = (
    jump_var
    /
    (
        jump_rv
        + eps
    )
)

### position Inside recent range 

In [51]:
rolling_high_300 = (
    df_1s
    .groupby(["symbol", "date"])["mid"]
    .transform(
        lambda x: x.rolling(
            300,
            min_periods=120
        ).max()
    )
)

rolling_low_300 = (
    df_1s
    .groupby(["symbol", "date"])["mid"]
    .transform(
        lambda x: x.rolling(
            300,
            min_periods=120
        ).min()
    )
)

df_1s["range_position_300s"] = (
    (
        df_1s["mid"]
        - rolling_low_300
    )
    /
    (
        rolling_high_300
        - rolling_low_300
        + eps
    )
)

In [52]:
CORE_FEATURES_V1 = [

    # =====================================================
    # ORDER FLOW
    # =====================================================

    "ofi_l1_sum_60s_signed_log",
    "ofi_rate_diff_30_300_sl",

    # =====================================================
    # ACTIVITY
    # =====================================================

    "ofi_activity_60s",
    "ofi_activity_300s",
    "update_rate_300s",
    "price_move_rate_30s",

    # =====================================================
    # PRICE IMPACT / RESPONSE
    # =====================================================

    "impact_lambda_1800s",
    "impact_r2_1800s",
    "ofi_response_gap_z",

    # =====================================================
    # RESTING LIQUIDITY
    # =====================================================

    "queue_imbalance_l1",
    "depth_imbalance_l5",
    "queue_replenishment_pressure_60s",
    "queue_imbalance_delta",
    "depth_imbalance_delta",

    # =====================================================
    # LIQUIDITY REGIME
    # =====================================================

    "spread_rank_1800s",

    # =====================================================
    # EXECUTED FLOW
    # =====================================================

    "volume_delta_sum_300s_sl",

    # =====================================================
    # PRICE DISPLACEMENT
    # =====================================================

    "price_mom_10s_new",
    "price_mom_30s_new",
    "price_mom_60s_new",

    # =====================================================
    # VOLATILITY
    # =====================================================

    "log_rv_300s_new",
    "rv_ratio_60_300_new",

    # =====================================================
    # LOCAL DEPENDENCE
    # =====================================================

    "ret_acf_lag1_300s",

    # =====================================================
    # VARIANCE RATIO
    # =====================================================

    "log_vr_5_300s",
    "log_vr_30_300s",

    # =====================================================
    # RETURN ASYMMETRY
    # =====================================================

    "semivar_imbalance_300s",
    "semivar_imbalance_900s",

    # =====================================================
    # DIRECTIONAL STRUCTURE
    # =====================================================

    "direction_bias_300s",
    "sign_persistence_900s",

    # =====================================================
    # JUMP REGIME
    # =====================================================

    "jump_ratio_300s",

    # =====================================================
    # LOCAL PRICE LOCATION
    # =====================================================

    "range_position_300s",
]

print(
    "Number of V1 core futures features:",
    len(CORE_FEATURES_V1)
)

assert len(CORE_FEATURES_V1) == 30

Number of V1 core futures features: 30


In [53]:
import numpy as np
import pandas as pd

# =========================================================
# FINAL FEATURE HEALTH CHECK
# =========================================================

FEATURES = CORE_FEATURES_V1

print("Number of features:", len(FEATURES))

# ---------------------------------------------------------
# 1. MISSINGNESS
# ---------------------------------------------------------

missing_summary = pd.DataFrame({
    "missing_count": df_1s[FEATURES].isna().sum(),
    "missing_pct": df_1s[FEATURES].isna().mean() * 100
}).sort_values("missing_pct", ascending=False)

print("\n===== MISSINGNESS =====")
print(missing_summary.round(3))

# ---------------------------------------------------------
# 2. INFINITIES
# ---------------------------------------------------------

inf_summary = pd.Series(
    {
        col: np.isinf(df_1s[col]).sum()
        for col in FEATURES
    },
    name="inf_count"
).sort_values(ascending=False)

print("\n===== INFINITE VALUES =====")
print(inf_summary)

# ---------------------------------------------------------
# 3. UNIQUE VALUES / CONSTANT FEATURES
# ---------------------------------------------------------

unique_summary = pd.DataFrame({
    "nunique": df_1s[FEATURES].nunique(dropna=True)
}).sort_values("nunique")

print("\n===== UNIQUE VALUES =====")
print(unique_summary)

# ---------------------------------------------------------
# 4. BASIC DISTRIBUTION
# ---------------------------------------------------------

print("\n===== FEATURE DISTRIBUTIONS =====")

print(
    df_1s[FEATURES]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
    .T
)

# ---------------------------------------------------------
# 5. EXTREME VALUE RATIO
#
# crude diagnostic:
# ratio of 99.9 percentile magnitude to median absolute value
# ---------------------------------------------------------

extreme_rows = []

for col in FEATURES:

    x = (
        df_1s[col]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    if len(x) == 0:
        continue

    median_abs = np.median(np.abs(x))

    q999 = np.quantile(
        np.abs(x),
        0.999
    )

    ratio = (
        q999
        /
        (median_abs + 1e-12)
    )

    extreme_rows.append({
        "feature": col,
        "median_abs": median_abs,
        "abs_q999": q999,
        "q999_to_median_ratio": ratio
    })

extreme_summary = (
    pd.DataFrame(extreme_rows)
    .sort_values(
        "q999_to_median_ratio",
        ascending=False
    )
)

print("\n===== EXTREME VALUE DIAGNOSTIC =====")
print(
    extreme_summary.head(20).round(4)
)

# ---------------------------------------------------------
# 6. PER-DAY VALID COVERAGE
# ---------------------------------------------------------

coverage = (
    df_1s
    .groupby("date")[FEATURES]
    .apply(
        lambda x: x.notna().mean()
    )
)

print("\n===== PER-DAY FEATURE COVERAGE =====")

print(
    coverage.round(3)
)

# ---------------------------------------------------------
# 7. ROW-LEVEL COMPLETE FEATURE AVAILABILITY
# ---------------------------------------------------------

df_1s["_n_valid_core_features"] = (
    df_1s[FEATURES]
    .notna()
    .sum(axis=1)
)

df_1s["_all_core_features_valid"] = (
    df_1s["_n_valid_core_features"]
    == len(FEATURES)
)

print("\n===== COMPLETE FEATURE ROWS =====")

print(
    df_1s["_all_core_features_valid"]
    .value_counts()
)

print(
    "\nFraction complete:",
    df_1s["_all_core_features_valid"].mean()
)

print(
    "\nValid feature count distribution:"
)

print(
    df_1s["_n_valid_core_features"]
    .describe()
)

Number of features: 30

===== MISSINGNESS =====
                                  missing_count  missing_pct
direction_bias_300s                      146063       27.857
sign_persistence_900s                     46162        8.804
ofi_response_gap_z                        23370        4.457
impact_lambda_1800s                       11970        2.283
impact_r2_1800s                           11970        2.283
ret_acf_lag1_300s                          5964        1.137
spread_rank_1800s                          5681        1.083
semivar_imbalance_900s                     5681        1.083
log_vr_30_300s                             2831        0.540
log_vr_5_300s                              2356        0.449
jump_ratio_300s                            2299        0.438
semivar_imbalance_300s                     2261        0.431
range_position_300s                        2261        0.431
ofi_rate_diff_30_300_sl                    1900        0.362
rv_ratio_60_300_new                  

In [54]:

import numpy as np
import pandas as pd

# ============================================================
# FEATURE 29 — SPREAD-DEPTH STRESS (900)
#
# Research reference:
#   Calendar experiments 21A–21G
#
# Selected specification:
#   Lookback: 900 observations
#   Minimum valid observations: 810
#   Current best bid/ask spread in bps
#   Trailing median five-level displayed depth
#   Square-root depth normalization
#   log1p transformation
#
# Output:
#   df_1s["spread_depth_stress_900"]
#
# No forward-looking data.
# No modification to existing engineered features.
# ============================================================

WINDOW = 900
MIN_PERIODS = 810

required = [
    "symbol",
    "date",
    "ts_sec",
    "bid_p1",
    "ask_p1",
    "bid_q1",
    "bid_q2",
    "bid_q3",
    "bid_q4",
    "bid_q5",
    "ask_q1",
    "ask_q2",
    "ask_q3",
    "ask_q4",
    "ask_q5"
]

missing = [
    col for col in required
    if col not in df_1s.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

if not df_1s.columns.is_unique:
    raise ValueError("Duplicate column names detected.")

if df_1s.duplicated(
    ["symbol", "ts_sec"]
).any():
    raise ValueError(
        "Duplicate symbol/timestamp records detected."
    )

# ------------------------------------------------------------
# 1. Build an independent calculation frame.
#
# This allows us to calculate the feature without
# changing other columns in the original df_1s.
# ------------------------------------------------------------

x_stress = df_1s[required].copy()

x_stress["ts_sec"] = pd.to_numeric(
    x_stress["ts_sec"], errors="raise"
)

x_stress = x_stress.sort_values(
    ["symbol", "date", "ts_sec"]
)

# ------------------------------------------------------------
# 2. Reconstruct current quoted spread (bps).
# ------------------------------------------------------------

bid = pd.to_numeric(
    x_stress["bid_p1"], errors="coerce"
)

ask = pd.to_numeric(
    x_stress["ask_p1"], errors="coerce"
)

mid = (bid + ask) / 2.0

valid_quote = (
    (bid > 0)
    & (ask > 0)
    & (ask >= bid)
    & np.isfinite(mid)
)

spread_bps = 10000.0 * (ask - bid) / mid

x_stress["_stress_spread_bps"] = (
    spread_bps.where(valid_quote)
)

# ------------------------------------------------------------
# 3. Calculate five-level displayed book depth.
# ------------------------------------------------------------

qty_cols = [
    "bid_q1",
    "bid_q2",
    "bid_q3",
    "bid_q4",
    "bid_q5",
    "ask_q1",
    "ask_q2",
    "ask_q3",
    "ask_q4",
    "ask_q5"
]

quantities = x_stress[qty_cols].apply(
    pd.to_numeric, errors="coerce"
)

valid_depth = (
    quantities.notna().all(axis=1)
    & np.isfinite(quantities).all(axis=1)
    & (quantities >= 0).all(axis=1)
)

x_stress["_stress_depth_l5"] = (
    quantities.sum(axis=1).where(valid_depth)
)

# ------------------------------------------------------------
# 4. Trailing 900-observation median depth.
#
# Same symbol and same session only.
# Current observation included.
#
# Matches the research construction.
# ------------------------------------------------------------

x_stress["_stress_depth_med_900"] = (
    x_stress
    .groupby(
        ["symbol", "date"],
        sort=False
    )["_stress_depth_l5"]
    .transform(
        lambda s: s.rolling(
            window=WINDOW,
            min_periods=MIN_PERIODS
        ).median()
    )
)

# ------------------------------------------------------------
# 5. Final feature.
# ------------------------------------------------------------

safe_spread = (
    x_stress["_stress_spread_bps"]
    .clip(lower=0)
)

safe_depth = (
    x_stress["_stress_depth_med_900"]
    .clip(lower=0)
)

x_stress["spread_depth_stress_900"] = np.log1p(
    safe_spread / np.sqrt(safe_depth + 1.0)
)

# ------------------------------------------------------------
# 6. Restore original row alignment.
#
# No master sorting or merge needed.
# ------------------------------------------------------------

new_feature = (
    x_stress["spread_depth_stress_900"]
    .reindex(df_1s.index)
)

if not df_1s.index.is_unique:
    raise ValueError(
        "Original df_1s index is not unique; "
        "use a key-based assignment instead."
    )

# Add only the new feature.
df_1s["spread_depth_stress_900"] = new_feature

# ------------------------------------------------------------
# 7. Validation.
# ------------------------------------------------------------

feature = df_1s["spread_depth_stress_900"]

if np.isinf(feature.to_numpy(dtype=float)).any():
    raise ValueError(
        "Infinite values detected in new feature."
    )

if (feature.dropna() < 0).any():
    raise ValueError(
        "Unexpected negative liquidity-stress values."
    )

# Compare with calculation output.
np.testing.assert_allclose(
    feature.to_numpy(dtype=float),
    new_feature.to_numpy(dtype=float),
    rtol=0,
    atol=1e-12,
    equal_nan=True
)

print("\n===== SPREAD-DEPTH STRESS 900 =====")

print("Lookback observations:", WINDOW)
print("Minimum valid observations:", MIN_PERIODS)
print("Total rows:", len(feature))
print("Valid rows:", feature.notna().sum())
print(
    "Missing percentage:",
    round(100 * feature.isna().mean(), 4)
)

print("\n===== DISTRIBUTION =====")

print(
    feature.describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)

print("\n===== FEATURE VALIDATION =====")
print("Infinite values: 0")
print("Existing features: unchanged")
print("Look-ahead leakage: none in construction")
print("New feature: spread_depth_stress_900")
print("Status: experimental / pending final OOS validation")


===== SPREAD-DEPTH STRESS 900 =====
Lookback observations: 900
Minimum valid observations: 810
Total rows: 524325
Valid rows: 508954
Missing percentage: 2.9316

===== DISTRIBUTION =====
count    508954.000000
mean          0.018188
std           0.014137
min           0.001697
1%            0.002639
5%            0.003739
25%           0.008354
50%           0.014501
75%           0.023664
95%           0.046239
99%           0.069150
max           0.519394
Name: spread_depth_stress_900, dtype: float64

===== FEATURE VALIDATION =====
Infinite values: 0
Existing features: unchanged
Look-ahead leakage: none in construction
New feature: spread_depth_stress_900
Status: experimental / pending final OOS validation


In [55]:

import numpy as np
import pandas as pd

# ============================================================
# FEATURE 30 — RECOVERY AVAILABILITY (300)
#
# Research reference:
#   Experiments 22E–22I
#
# Definition:
#   300-observation bid/ask recovery eligibility flag
#
# Parameters:
#   Reference depth window = 300 observations
#   Reference minimum observations = 270
#   Depletion threshold = 0.75
#   Availability window = 300 observations
#   Minimum valid transitions = 270
#   Minimum depleted transitions per side = 5
#
# Output:
#   df_1s["recovery_available_300"]
#
# Uses current and historical observations only.
# No change to existing feature columns.
# ============================================================

REFERENCE_WINDOW = 300
REFERENCE_MIN = 270

RECOVERY_WINDOW = 300
MIN_VALID_TRANSITIONS = 270
MIN_DEPLETED_EVENTS = 5
DEPLETION_THRESHOLD = 0.75

required = [
    "symbol",
    "ts_sec",
    "bid_q1", "bid_q2", "bid_q3", "bid_q4", "bid_q5",
    "ask_q1", "ask_q2", "ask_q3", "ask_q4", "ask_q5"
]

missing = [
    c for c in required
    if c not in df_1s.columns
]

if missing:
    raise ValueError(
        f"Missing required fields: {missing}"
    )

if not df_1s.columns.is_unique:
    raise ValueError("Duplicate column names found.")

# Independent calculation frame.
work = df_1s[required].copy()

work["_original_position"] = np.arange(len(work))

work["ts_sec"] = pd.to_numeric(
    work["ts_sec"], errors="raise"
)

if work.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError(
        "Duplicate symbol/timestamp observations."
    )

# Derive IST date directly from timestamps, matching 22E.
ts_ist = pd.to_datetime(
    work["ts_sec"], unit="s", utc=True
).dt.tz_convert("Asia/Kolkata")

work["_session_date"] = ts_ist.dt.date

work = work.sort_values(
    ["symbol", "_session_date", "ts_sec"]
).reset_index(drop=True)

GROUP = ["symbol", "_session_date"]

# ------------------------------------------------------------
# 1. Compute side-specific L5 displayed depth
# ------------------------------------------------------------

for side in ["bid", "ask"]:

    cols = [
        f"{side}_q{i}"
        for i in range(1, 6)
    ]

    q = work[cols].apply(
        pd.to_numeric, errors="coerce"
    ).replace(
        [np.inf, -np.inf], np.nan
    )

    valid_depth = (
        q.notna().all(axis=1)
        & q.ge(0).all(axis=1)
    )

    work[f"_{side}_depth"] = (
        q.sum(axis=1).where(valid_depth)
    )

# ------------------------------------------------------------
# 2. Causal previous-300-observation median reference
# ------------------------------------------------------------

for side in ["bid", "ask"]:

    depth_col = f"_{side}_depth"
    reference_col = f"_{side}_reference"
    relative_col = f"_{side}_relative"

    work[reference_col] = (
        work.groupby(GROUP, sort=False)[depth_col]
        .transform(
            lambda s: s.shift(1).rolling(
                window=REFERENCE_WINDOW,
                min_periods=REFERENCE_MIN
            ).median()
        )
    )

    work[relative_col] = (
        work[depth_col]
        / work[reference_col].where(
            work[reference_col] > 0
        )
    )

# ------------------------------------------------------------
# 3. Identify valid consecutive-second transitions
# ------------------------------------------------------------

previous_ts = (
    work.groupby(GROUP, sort=False)["ts_sec"]
    .shift(1)
)

consecutive = (
    work["ts_sec"] - previous_ts
).eq(1)

for side in ["bid", "ask"]:

    relative_col = f"_{side}_relative"

    current = work[relative_col]

    previous = (
        work.groupby(GROUP, sort=False)[relative_col]
        .shift(1)
    )

    valid_pair = (
        consecutive
        & np.isfinite(current)
        & np.isfinite(previous)
    )

    depleted_before = (
        previous < DEPLETION_THRESHOLD
    )

    work[f"_{side}_valid"] = (
        valid_pair.astype(np.float64)
    )

    work[f"_{side}_depleted"] = (
        (valid_pair & depleted_before)
        .astype(np.float64)
    )

# ------------------------------------------------------------
# 4. Count valid and depleted transitions over 300 rows
# ------------------------------------------------------------

for side in ["bid", "ask"]:

    grouped = work.groupby(GROUP, sort=False)

    work[f"_{side}_valid_count"] = (
        grouped[f"_{side}_valid"].transform(
            lambda s: s.rolling(
                window=RECOVERY_WINDOW,
                min_periods=1
            ).sum()
        )
    )

    work[f"_{side}_event_count"] = (
        grouped[f"_{side}_depleted"].transform(
            lambda s: s.rolling(
                window=RECOVERY_WINDOW,
                min_periods=1
            ).sum()
        )
    )

    work[f"_{side}_eligible"] = (
        work[f"_{side}_valid_count"].ge(
            MIN_VALID_TRANSITIONS
        )
        &
        work[f"_{side}_event_count"].ge(
            MIN_DEPLETED_EVENTS
        )
    )

# ------------------------------------------------------------
# 5. Final binary feature
# ------------------------------------------------------------

work["recovery_available_300"] = (
    work["_bid_eligible"]
    & work["_ask_eligible"]
).astype(np.int8)

# ------------------------------------------------------------
# 6. Restore original row order safely
# ------------------------------------------------------------

work = work.sort_values(
    "_original_position"
)

feature_values = (
    work["recovery_available_300"]
    .to_numpy(dtype=np.int8)
)

assert len(feature_values) == len(df_1s)

# Assign positionally; works with non-unique dataframe index.
df_1s["recovery_available_300"] = feature_values

# ------------------------------------------------------------
# 7. Health checks
# ------------------------------------------------------------

feature = df_1s["recovery_available_300"]

if not feature.isin([0, 1]).all():
    raise ValueError(
        "Recovery availability must be binary."
    )

print("\n===== RECOVERY AVAILABILITY 300 =====")

print("Total rows:", len(feature))
print("Flagged 1:", int((feature == 1).sum()))
print("Flagged 0:", int((feature == 0).sum()))

print("\n===== DISTRIBUTION =====")

print(
    feature.value_counts(
        normalize=True
    ).mul(100).round(4)
)

print("\n===== FEATURE VALIDATION =====")

print("Unique values:", sorted(feature.unique()))
print("Missing values:", int(feature.isna().sum()))
print("Current/future leakage: none in construction")
print("Existing feature columns unchanged")
print("Status: provisional / pending full OOS validation")


===== RECOVERY AVAILABILITY 300 =====
Total rows: 524325
Flagged 1: 350829
Flagged 0: 173496

===== DISTRIBUTION =====
recovery_available_300
1    66.9106
0    33.0894
Name: proportion, dtype: float64

===== FEATURE VALIDATION =====
Unique values: [np.int8(0), np.int8(1)]
Missing values: 0
Current/future leakage: none in construction
Existing feature columns unchanged
Status: provisional / pending full OOS validation


In [57]:

import numpy as np
import pandas as pd

# ============================================================
# FEATURE 31 — OFI / PRICE WEIGHTED DISAGREEMENT (60s)
#
# Research: Experiments 23A–23D
#
# Output:
#   df_1s["ofi_disagreement_weighted_60"]
#
# Specification:
#   Quote-based Level-1 OFI
#   Lookback = 60 observations
#   Minimum valid increments = 54
#   Exact 60-second historical momentum
#   Trailing 60-observation realized volatility
#   Volatility floor = 0.1 bps
#   Maximum relative-move weight = 3.0
#
# No future data.
# No change to existing feature columns.
# ============================================================

WINDOW_31 = 60
MIN_PERIODS_31 = 54
VOL_FLOOR_BPS_31 = 0.1
MAX_WEIGHT_31 = 3.0

OUTPUT_31 = "ofi_disagreement_weighted_60"

required_31 = [
    "symbol",
    "ts_sec",
    "bid_p1",
    "ask_p1",
    "bid_q1",
    "ask_q1",
]

missing_31 = [
    c for c in required_31
    if c not in df_1s.columns
]

if missing_31:
    raise ValueError(
        f"Missing columns: {missing_31}"
    )

if not df_1s.columns.is_unique:
    raise ValueError("Duplicate dataframe columns.")

# ------------------------------------------------------------
# 1. Independent working frame, positional row identifier
# ------------------------------------------------------------

work_31 = df_1s[required_31].copy()

work_31["_original_position"] = np.arange(
    len(work_31)
)

for c in [
    "ts_sec",
    "bid_p1",
    "ask_p1",
    "bid_q1",
    "ask_q1",
]:
    work_31[c] = pd.to_numeric(
        work_31[c],
        errors="coerce"
    ).replace(
        [np.inf, -np.inf], np.nan
    )

if work_31["ts_sec"].isna().any():
    raise ValueError("Invalid ts_sec values.")

if work_31.duplicated(
    ["symbol", "ts_sec"]
).any():
    raise ValueError(
        "Duplicate symbol/timestamp records."
    )

ts_ist_31 = pd.to_datetime(
    work_31["ts_sec"],
    unit="s",
    utc=True
).dt.tz_convert("Asia/Kolkata")

work_31["_session_date"] = ts_ist_31.dt.date

work_31 = work_31.sort_values(
    ["symbol", "_session_date", "ts_sec"]
).reset_index(drop=True)

GROUP_31 = ["symbol", "_session_date"]

# ------------------------------------------------------------
# 2. Validate current Level-1 quotes
# ------------------------------------------------------------

bid_p = work_31["bid_p1"]
ask_p = work_31["ask_p1"]
bid_q = work_31["bid_q1"]
ask_q = work_31["ask_q1"]

valid_quote_31 = (
    bid_p.gt(0)
    & ask_p.gt(0)
    & ask_p.ge(bid_p)
    & bid_q.ge(0)
    & ask_q.ge(0)
    & np.isfinite(bid_p)
    & np.isfinite(ask_p)
    & np.isfinite(bid_q)
    & np.isfinite(ask_q)
)

work_31["_mid"] = (
    (bid_p + ask_p) / 2
).where(valid_quote_31)

# ------------------------------------------------------------
# 3. Previous Level-1 quote
# ------------------------------------------------------------

grouped_31 = work_31.groupby(
    GROUP_31,
    sort=False
)

prev_ts = grouped_31["ts_sec"].shift(1)
prev_bid_p = grouped_31["bid_p1"].shift(1)
prev_ask_p = grouped_31["ask_p1"].shift(1)
prev_bid_q = grouped_31["bid_q1"].shift(1)
prev_ask_q = grouped_31["ask_q1"].shift(1)

valid_prev_31 = (
    prev_bid_p.gt(0)
    & prev_ask_p.gt(0)
    & prev_ask_p.ge(prev_bid_p)
    & prev_bid_q.ge(0)
    & prev_ask_q.ge(0)
    & np.isfinite(prev_bid_p)
    & np.isfinite(prev_ask_p)
    & np.isfinite(prev_bid_q)
    & np.isfinite(prev_ask_q)
)

valid_pair_31 = (
    valid_quote_31
    & valid_prev_31
    & (work_31["ts_sec"] - prev_ts).eq(1)
)

# ------------------------------------------------------------
# 4. Level-1 quote-based OFI increment
# ------------------------------------------------------------

bid_contribution_31 = (
    (bid_p >= prev_bid_p).astype(float) * bid_q
    - (bid_p <= prev_bid_p).astype(float) * prev_bid_q
)

ask_contribution_31 = (
    -(ask_p <= prev_ask_p).astype(float) * ask_q
    + (ask_p >= prev_ask_p).astype(float) * prev_ask_q
)

work_31["_ofi"] = (
    bid_contribution_31 + ask_contribution_31
).where(valid_pair_31)

work_31["_abs_ofi"] = work_31["_ofi"].abs()

# ------------------------------------------------------------
# 5. Rolling 60-observation normalized OFI pressure
# ------------------------------------------------------------

grouped_31 = work_31.groupby(
    GROUP_31,
    sort=False
)

work_31["_ofi_sum_60"] = (
    grouped_31["_ofi"].transform(
        lambda s: s.rolling(
            window=WINDOW_31,
            min_periods=MIN_PERIODS_31
        ).sum()
    )
)

work_31["_ofi_gross_60"] = (
    grouped_31["_abs_ofi"].transform(
        lambda s: s.rolling(
            window=WINDOW_31,
            min_periods=MIN_PERIODS_31
        ).sum()
    )
)

gross = work_31["_ofi_gross_60"]
net = work_31["_ofi_sum_60"]

work_31["_pressure_60"] = (
    net / gross.where(gross > 0)
)

# Observed zero-flow window -> pressure = 0.
# Insufficient history stays missing.
zero_flow = gross.eq(0) & net.notna()

work_31.loc[
    zero_flow,
    "_pressure_60"
] = 0.0

# ------------------------------------------------------------
# 6. Exact 60-second historical momentum
# ------------------------------------------------------------

grouped_31 = work_31.groupby(
    GROUP_31,
    sort=False
)

old_mid = grouped_31["_mid"].shift(WINDOW_31)
old_ts = grouped_31["ts_sec"].shift(WINDOW_31)

valid_historical_endpoint = (
    (work_31["ts_sec"] - old_ts).eq(WINDOW_31)
    & work_31["_mid"].gt(0)
    & old_mid.gt(0)
)

work_31["_momentum_60"] = (
    1e4 * np.log(
        work_31["_mid"] / old_mid
    )
).where(valid_historical_endpoint)

# ------------------------------------------------------------
# 7. Trailing 60-observation realized volatility
# ------------------------------------------------------------

previous_mid = grouped_31["_mid"].shift(1)

one_sec_log_return = np.log(
    work_31["_mid"] / previous_mid
).where(valid_pair_31)

work_31["_ret_sq"] = (
    one_sec_log_return ** 2
)

work_31["_rv_60"] = (
    work_31.groupby(
        GROUP_31,
        sort=False
    )["_ret_sq"]
    .transform(
        lambda s: s.rolling(
            window=WINDOW_31,
            min_periods=MIN_PERIODS_31
        ).sum()
    )
    .clip(lower=0)
    .pow(0.5)
    * 1e4
)

# ------------------------------------------------------------
# 8. Construct directional disagreement
# ------------------------------------------------------------

pressure = work_31["_pressure_60"]
momentum = work_31["_momentum_60"]
rv = work_31["_rv_60"]

valid_components = (
    np.isfinite(pressure)
    & np.isfinite(momentum)
    & np.isfinite(rv)
)

disagreement = (
    (pressure * momentum) < 0
)

relative_move_weight = (
    momentum.abs()
    / rv.clip(lower=VOL_FLOOR_BPS_31)
).clip(upper=MAX_WEIGHT_31)

work_31[OUTPUT_31] = (
    (pressure * relative_move_weight)
    .where(disagreement, 0.0)
    .where(valid_components)
)

# ------------------------------------------------------------
# 9. Restore original row order
# ------------------------------------------------------------

work_31 = work_31.sort_values(
    "_original_position"
)

feature_values_31 = (
    work_31[OUTPUT_31]
    .to_numpy(dtype=float)
)

if len(feature_values_31) != len(df_1s):
    raise ValueError("Unexpected feature length.")

# Only new output column is assigned to df_1s.
df_1s[OUTPUT_31] = feature_values_31

# ------------------------------------------------------------
# 10. Feature validation
# ------------------------------------------------------------

feature_31 = df_1s[OUTPUT_31]

if np.isinf(feature_31.to_numpy()).any():
    raise ValueError("Infinite values detected.")

if (feature_31.abs().dropna() > MAX_WEIGHT_31 + 1e-9).any():
    raise ValueError(
        "Unexpected value outside theoretical range."
    )

print("\n===== FEATURE 31 — OFI DISAGREEMENT =====")

print("Feature:", OUTPUT_31)
print("Total rows:", len(feature_31))
print("Valid rows:", int(feature_31.notna().sum()))
print(
    "Valid percentage:",
    round(feature_31.notna().mean() * 100, 4)
)
print(
    "Zero percentage:",
    round(feature_31.eq(0).mean() * 100, 4)
)

print("\n===== DISTRIBUTION =====")

print(
    feature_31.describe(
        percentiles=[
            0.01, 0.05, 0.25,
            0.50, 0.75, 0.95, 0.99
        ]
    )
)

print("\n===== VALIDATION =====")
print("Infinite values: 0")
print("Theoretical range: [-3, 3]")
print("Session boundaries: respected")
print("Exact historical momentum endpoints: checked")
print("Original features: unchanged")
print("Status: experimental, pending future validation")



===== FEATURE 31 — OFI DISAGREEMENT =====
Feature: ofi_disagreement_weighted_60
Total rows: 524325
Valid rows: 523185
Valid percentage: 99.7826
Zero percentage: 85.3926

===== DISTRIBUTION =====
count    523185.000000
mean          0.001749
std           0.111524
min          -1.290043
1%           -0.367236
5%           -0.024216
25%           0.000000
50%           0.000000
75%           0.000000
95%           0.034378
99%           0.427463
max           1.477476
Name: ofi_disagreement_weighted_60, dtype: float64

===== VALIDATION =====
Infinite values: 0
Theoretical range: [-3, 3]
Session boundaries: respected
Exact historical momentum endpoints: checked
Original features: unchanged
Status: experimental, pending future validation


In [65]:

import numpy as np
import pandas as pd

# ============================================================
# FEATURE 32 — QUOTE RETREAT IMBALANCE (30s)
#
# Based on Experiment 24A
# Confirmed exploratory usefulness: Experiments 24B–24D
#
# Output:
#   df_1s["quote_retreat_imbalance_30"]
#
# Window: 30 trailing observations
# Minimum valid transitions: 27
#
# Bid retreat: best bid decreases
# Ask retreat: best ask increases
#
# Positive = ask retreats dominate
# Negative = bid retreats dominate
#
# No future information.
# No modification to existing feature columns.
# ============================================================

WINDOW_32 = 30
MIN_VALID_32 = 27

FEATURE_32 = "quote_retreat_imbalance_30"

REQUIRED_32 = [
    "symbol",
    "ts_sec",
    "bid_p1",
    "ask_p1"
]

missing_32 = [
    col for col in REQUIRED_32
    if col not in df_1s.columns
]

if missing_32:
    raise ValueError(
        f"Missing required columns: {missing_32}"
    )

if not df_1s.columns.is_unique:
    raise ValueError("Duplicate DataFrame columns.")

# ------------------------------------------------------------
# 1. Independent working DataFrame
# ------------------------------------------------------------

work_32 = df_1s[REQUIRED_32].copy()

work_32["_original_position"] = np.arange(
    len(work_32)
)

for col in ["ts_sec", "bid_p1", "ask_p1"]:

    work_32[col] = pd.to_numeric(
        work_32[col],
        errors="coerce"
    ).replace(
        [np.inf, -np.inf],
        np.nan
    )

if work_32["ts_sec"].isna().any():
    raise ValueError("Invalid timestamps.")

if work_32.duplicated(
    ["symbol", "ts_sec"]
).any():
    raise ValueError(
        "Duplicate symbol/timestamp observations."
    )

# ------------------------------------------------------------
# 2. Convert timestamps to IST trading dates
#
# All rolling calculations are strictly separated
# by symbol and IST session date.
# ------------------------------------------------------------

ts_ist_32 = pd.to_datetime(
    work_32["ts_sec"],
    unit="s",
    utc=True
).dt.tz_convert("Asia/Kolkata")

work_32["_session_date"] = (
    ts_ist_32.dt.date
)

work_32 = work_32.sort_values(
    ["symbol", "_session_date", "ts_sec"]
).reset_index(drop=True)

GROUP_32 = [
    "symbol",
    "_session_date"
]

# ------------------------------------------------------------
# 3. Current and previous best bid/ask quotes
# ------------------------------------------------------------

bid_32 = work_32["bid_p1"]
ask_32 = work_32["ask_p1"]

valid_quote_32 = (
    bid_32.gt(0)
    & ask_32.gt(0)
    & ask_32.ge(bid_32)
    & np.isfinite(bid_32)
    & np.isfinite(ask_32)
)

grouped_32 = work_32.groupby(
    GROUP_32,
    sort=False
)

prev_ts_32 = grouped_32["ts_sec"].shift(1)
prev_bid_32 = grouped_32["bid_p1"].shift(1)
prev_ask_32 = grouped_32["ask_p1"].shift(1)

valid_prev_32 = (
    prev_bid_32.gt(0)
    & prev_ask_32.gt(0)
    & prev_ask_32.ge(prev_bid_32)
    & np.isfinite(prev_bid_32)
    & np.isfinite(prev_ask_32)
)

# Exactly consecutive 1-second observations.
valid_pair_32 = (
    valid_quote_32
    & valid_prev_32
    & (
        work_32["ts_sec"] - prev_ts_32
    ).eq(1)
)

# ------------------------------------------------------------
# 4. Define retreat events
#
# Invalid transitions are excluded from numerator.
# ------------------------------------------------------------

work_32["_valid_transition"] = (
    valid_pair_32.astype(float)
)

work_32["_bid_retreat"] = (
    bid_32.lt(prev_bid_32).astype(float)
).where(valid_pair_32)

work_32["_ask_retreat"] = (
    ask_32.gt(prev_ask_32).astype(float)
).where(valid_pair_32)

# Positive = ask retreats dominate.
work_32["_signed_retreat"] = (
    work_32["_ask_retreat"]
    - work_32["_bid_retreat"]
)

# ------------------------------------------------------------
# 5. Rolling 30-observation sums
#
# Matches Experiment 24A:
#   - rolling numerator with invalids filled as 0
#   - denominator = number of valid transitions
#   - min 27 valid transitions
# ------------------------------------------------------------

grouped_32 = work_32.groupby(
    GROUP_32,
    sort=False
)

valid_count_32 = (
    grouped_32["_valid_transition"]
    .transform(
        lambda s: s.rolling(
            window=WINDOW_32,
            min_periods=1
        ).sum()
    )
)

retreat_sum_32 = (
    grouped_32["_signed_retreat"]
    .transform(
        lambda s: s.fillna(0.0).rolling(
            window=WINDOW_32,
            min_periods=1
        ).sum()
    )
)

# ------------------------------------------------------------
# 6. Final quote retreat imbalance
# ------------------------------------------------------------

denominator_32 = valid_count_32.where(
    valid_count_32 >= MIN_VALID_32
)

work_32[FEATURE_32] = (
    retreat_sum_32 / denominator_32
)

# ------------------------------------------------------------
# 7. Restore original row order
# ------------------------------------------------------------

work_32 = work_32.sort_values(
    "_original_position"
)

feature_values_32 = (
    work_32[FEATURE_32]
    .to_numpy(dtype=float)
)

if len(feature_values_32) != len(df_1s):
    raise ValueError("Unexpected feature length.")

# Assign ONLY the new feature.
df_1s[FEATURE_32] = feature_values_32

# ------------------------------------------------------------
# 8. Validation
# ------------------------------------------------------------

feature_32 = df_1s[FEATURE_32]

if np.isinf(feature_32.to_numpy()).any():
    raise ValueError(
        "Infinite feature values detected."
    )

if (
    feature_32.dropna().abs() > 1.0 + 1e-12
).any():
    raise ValueError(
        "Quote retreat imbalance outside [-1, 1]."
    )

print("\n===== FEATURE 32 VALIDATION =====")

print("Feature:", FEATURE_32)
print("Total rows:", len(feature_32))

print(
    "Valid rows:",
    int(feature_32.notna().sum())
)

print(
    "Valid percentage:",
    round(
        100 * feature_32.notna().mean(), 4
    )
)

print(
    "Zero percentage:",
    round(
        100 * feature_32.eq(0).mean(), 4
    )
)

print("\n===== FEATURE DISTRIBUTION =====")

print(
    feature_32.describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)

print("\n===== FEATURE STATUS =====")
print("Causal calculation: yes")
print("Session boundary grouping: yes")
print("Original feature columns unchanged")
print("Research status: experimental")


===== FEATURE 32 VALIDATION =====
Feature: quote_retreat_imbalance_30
Total rows: 524325
Valid rows: 523812
Valid percentage: 99.9022
Zero percentage: 56.2661

===== FEATURE DISTRIBUTION =====
count    523812.000000
mean          0.006432
std           0.048611
min          -0.407407
1%           -0.100000
5%           -0.033333
25%           0.000000
50%           0.000000
75%           0.000000
95%           0.066667
99%           0.200000
max           0.600000
Name: quote_retreat_imbalance_30, dtype: float64

===== FEATURE STATUS =====
Causal calculation: yes
Session boundary grouping: yes
Original feature columns unchanged
Research status: experimental


In [67]:

import numpy as np
import pandas as pd

# ============================================================
# FEATURES #33 AND #34 — ORDER-BOOK GEOMETRY
#
# Exact definitions from Experiment 28A
#
# Output:
#   df_1s["slope_asymmetry"]
#   df_1s["curvature_asymmetry"]
#
# No rolling windows or future information.
# No modification to existing features.
# ============================================================

bid_price_cols = [f"bid_p{i}" for i in range(1, 6)]
ask_price_cols = [f"ask_p{i}" for i in range(1, 6)]
bid_qty_cols = [f"bid_q{i}" for i in range(1, 6)]
ask_qty_cols = [f"ask_q{i}" for i in range(1, 6)]

required = (
    bid_price_cols + ask_price_cols
    + bid_qty_cols + ask_qty_cols
)

missing = [c for c in required if c not in df_1s.columns]

if missing:
    raise ValueError(f"Missing book columns: {missing}")

work = df_1s[required].apply(
    pd.to_numeric, errors="coerce"
)

BP = work[bid_price_cols].to_numpy(dtype=np.float64)
AP = work[ask_price_cols].to_numpy(dtype=np.float64)
BQ = work[bid_qty_cols].to_numpy(dtype=np.float64)
AQ = work[ask_qty_cols].to_numpy(dtype=np.float64)

mid = (BP[:, 0] + AP[:, 0]) / 2.0

# ------------------------------------------------------------
# 1. Strict book validation
# ------------------------------------------------------------

valid_prices = (
    np.isfinite(BP).all(axis=1)
    & np.isfinite(AP).all(axis=1)
    & (BP > 0).all(axis=1)
    & (AP > 0).all(axis=1)
    & (AP[:, 0] >= BP[:, 0])
    & (np.diff(BP, axis=1) < 0).all(axis=1)
    & (np.diff(AP, axis=1) > 0).all(axis=1)
)

valid_quantities = (
    np.isfinite(BQ).all(axis=1)
    & np.isfinite(AQ).all(axis=1)
    & (BQ >= 0).all(axis=1)
    & (AQ >= 0).all(axis=1)
    & (BQ.sum(axis=1) > 0)
    & (AQ.sum(axis=1) > 0)
)

valid_book = valid_prices & valid_quantities

# ------------------------------------------------------------
# 2. Price distances measured in basis points
# ------------------------------------------------------------

with np.errstate(divide="ignore", invalid="ignore"):
    DB = 1e4 * (BP[:, [0]] - BP) / mid[:, None]
    DA = 1e4 * (AP - AP[:, [0]]) / mid[:, None]

# ------------------------------------------------------------
# 3. Rowwise OLS slope with intercept
# ------------------------------------------------------------

def ols_slope_33_34(y, d):

    dx = d - d.mean(axis=1, keepdims=True)
    dy = y - y.mean(axis=1, keepdims=True)

    numerator = np.sum(dx * dy, axis=1)
    denominator = np.sum(dx * dx, axis=1)

    with np.errstate(divide="ignore", invalid="ignore"):
        return numerator / np.where(
            denominator > 1e-16,
            denominator,
            np.nan
        )

# ------------------------------------------------------------
# 4. Feature #33: normalized cumulative-depth slope
# ------------------------------------------------------------

CB = np.cumsum(BQ, axis=1)
CA = np.cumsum(AQ, axis=1)

with np.errstate(divide="ignore", invalid="ignore"):
    slope_bid = ols_slope_33_34(CB, DB) / CB[:, -1]
    slope_ask = ols_slope_33_34(CA, DA) / CA[:, -1]

slope_asymmetry = slope_bid - slope_ask

# ------------------------------------------------------------
# 5. Feature #34: liquidity-density gradient asymmetry
# ------------------------------------------------------------

gap_bid = np.diff(DB, axis=1)
gap_ask = np.diff(DA, axis=1)

with np.errstate(divide="ignore", invalid="ignore"):

    density_bid = BQ[:, 1:] / np.where(
        gap_bid > 0, gap_bid, np.nan
    )

    density_ask = AQ[:, 1:] / np.where(
        gap_ask > 0, gap_ask, np.nan
    )

curvature_bid = ols_slope_33_34(
    np.log1p(density_bid),
    DB[:, 1:]
)

curvature_ask = ols_slope_33_34(
    np.log1p(density_ask),
    DA[:, 1:]
)

curvature_asymmetry = curvature_bid - curvature_ask

# ------------------------------------------------------------
# 6. Match Experiment 28A invalid-book handling
# ------------------------------------------------------------

slope_asymmetry[~valid_book] = np.nan
curvature_asymmetry[~valid_book] = np.nan

slope_asymmetry[~np.isfinite(slope_asymmetry)] = np.nan
curvature_asymmetry[~np.isfinite(curvature_asymmetry)] = np.nan

# ------------------------------------------------------------
# 7. Assign ONLY two new features
# ------------------------------------------------------------

df_1s["slope_asymmetry"] = slope_asymmetry
df_1s["curvature_asymmetry"] = curvature_asymmetry

# ------------------------------------------------------------
# 8. Validation
# ------------------------------------------------------------

print("===== FEATURE 33 AND 34 VALIDATION =====")
print("Total rows:", len(df_1s))
print("Valid five-level books:", int(valid_book.sum()))

for name in [
    "slope_asymmetry",
    "curvature_asymmetry"
]:
    s = df_1s[name]

    print(f"\n{name}")
    print("Valid %:", round(100 * s.notna().mean(), 4))
    print(s.describe(percentiles=[0.05, 0.5, 0.95]))

print("\nExisting feature columns untouched.")
print("Features #33 and #34 calculated.")


===== FEATURE 33 AND 34 VALIDATION =====
Total rows: 524325
Valid five-level books: 524325

slope_asymmetry
Valid %: 100.0
count    524325.000000
mean          0.009726
std           0.279559
min          -1.051163
5%           -0.459508
50%           0.014642
95%           0.465293
max           1.099917
Name: slope_asymmetry, dtype: float64

curvature_asymmetry
Valid %: 100.0
count    524325.000000
mean          0.325263
std           2.653782
min         -13.144028
5%           -3.879479
50%           0.231492
95%           4.810746
max          13.846560
Name: curvature_asymmetry, dtype: float64

Existing feature columns untouched.
Features #33 and #34 calculated.


In [68]:

import numpy as np
import pandas as pd

# ============================================================
# FEATURE #35 — WEIGHTED GAP ASYMMETRY
# FEATURE #36 — FRAGILITY ASYMMETRY
#
# Exact formulas from Experiment 29A.
#
# Input:
#   df_1s with five bid and ask price levels
#   and five bid and ask quantity levels.
#
# Output:
#   df_1s["weighted_gap_asymmetry"]
#   df_1s["fragility_asymmetry"]
#
# Causal snapshot features.
# Existing columns are preserved.
# ============================================================

LAMBDA_GAP = 0.7

# ------------------------------------------------------------
# 1. Required columns
# ------------------------------------------------------------

bid_price_cols = [
    f"bid_p{i}" for i in range(1, 6)
]

ask_price_cols = [
    f"ask_p{i}" for i in range(1, 6)
]

bid_qty_cols = [
    f"bid_q{i}" for i in range(1, 6)
]

ask_qty_cols = [
    f"ask_q{i}" for i in range(1, 6)
]

required_cols = (
    bid_price_cols
    + ask_price_cols
    + bid_qty_cols
    + ask_qty_cols
)

missing_cols = [
    c for c in required_cols
    if c not in df_1s.columns
]

if missing_cols:
    raise ValueError(
        f"Missing required columns: {missing_cols}"
    )

# ------------------------------------------------------------
# 2. Extract five-level order book
# ------------------------------------------------------------

work = df_1s[required_cols].apply(
    pd.to_numeric,
    errors="coerce"
)

work = work.replace(
    [np.inf, -np.inf],
    np.nan
)

BP = work[bid_price_cols].to_numpy(
    dtype=np.float64
)

AP = work[ask_price_cols].to_numpy(
    dtype=np.float64
)

BQ = work[bid_qty_cols].to_numpy(
    dtype=np.float64
)

AQ = work[ask_qty_cols].to_numpy(
    dtype=np.float64
)

mid = (
    BP[:, 0] + AP[:, 0]
) / 2.0

# ------------------------------------------------------------
# 3. Validate the order book
# ------------------------------------------------------------

valid_book = (
    np.isfinite(BP).all(axis=1)
    & np.isfinite(AP).all(axis=1)
    & np.isfinite(BQ).all(axis=1)
    & np.isfinite(AQ).all(axis=1)
    & (BP > 0).all(axis=1)
    & (AP > 0).all(axis=1)
    & (BQ >= 0).all(axis=1)
    & (AQ >= 0).all(axis=1)
    & (BP[:, 0] <= AP[:, 0])
    & (np.diff(BP, axis=1) < 0).all(axis=1)
    & (np.diff(AP, axis=1) > 0).all(axis=1)
    & (BQ.sum(axis=1) > 0)
    & (AQ.sum(axis=1) > 0)
)

# ------------------------------------------------------------
# 4. Adjacent level gaps in basis points
#
# Bid:
#   gap_i = bid_price_i - bid_price_(i+1)
#
# Ask:
#   gap_i = ask_price_(i+1) - ask_price_i
# ------------------------------------------------------------

with np.errstate(
    divide="ignore",
    invalid="ignore"
):

    gap_bid = (
        -np.diff(BP, axis=1)
        / mid[:, None]
        * 1e4
    )

    gap_ask = (
        np.diff(AP, axis=1)
        / mid[:, None]
        * 1e4
    )

gap_bid[~valid_book] = np.nan
gap_ask[~valid_book] = np.nan

# ------------------------------------------------------------
# 5. Exponential distance weights
#
# w_i proportional to exp(-0.7 * (i-1))
# ------------------------------------------------------------

level_index = np.arange(
    4,
    dtype=np.float64
)

weights = np.exp(
    -LAMBDA_GAP * level_index
)

weights = weights / weights.sum()

print("===== GAP WEIGHTS =====")

for i, weight in enumerate(weights, start=1):
    print(
        f"Level {i}->{i+1}: {weight:.6f}"
    )

# ------------------------------------------------------------
# 6. FEATURE #35
#
# Weighted gap asymmetry
# ------------------------------------------------------------

weighted_bid_gap = gap_bid @ weights
weighted_ask_gap = gap_ask @ weights

weighted_gap_asymmetry = (
    weighted_bid_gap - weighted_ask_gap
)

# ------------------------------------------------------------
# 7. FEATURE #36
#
# Fragility asymmetry
#
# Quantity at level i lies immediately before
# the price gap between levels i and i+1.
# ------------------------------------------------------------

with np.errstate(
    divide="ignore",
    invalid="ignore"
):

    bid_fragility = (
        gap_bid
        / np.sqrt(1.0 + BQ[:, :4])
    ) @ weights

    ask_fragility = (
        gap_ask
        / np.sqrt(1.0 + AQ[:, :4])
    ) @ weights

fragility_asymmetry = (
    bid_fragility - ask_fragility
)

# ------------------------------------------------------------
# 8. Final validity checks
# ------------------------------------------------------------

weighted_gap_asymmetry[~valid_book] = np.nan
fragility_asymmetry[~valid_book] = np.nan

weighted_gap_asymmetry[
    ~np.isfinite(weighted_gap_asymmetry)
] = np.nan

fragility_asymmetry[
    ~np.isfinite(fragility_asymmetry)
] = np.nan

# ------------------------------------------------------------
# 9. Assign only two new columns
#
# Assignment by row order, preserves df_1s index.
# ------------------------------------------------------------

df_1s["weighted_gap_asymmetry"] = (
    weighted_gap_asymmetry
)

df_1s["fragility_asymmetry"] = (
    fragility_asymmetry
)

# ------------------------------------------------------------
# 10. Diagnostics
# ------------------------------------------------------------

print("\n===== FEATURE #35 AND #36 VALIDATION =====")

print("Total rows:", len(df_1s))
print("Valid books:", int(valid_book.sum()))

print(
    "Valid books (%):",
    round(100 * valid_book.mean(), 4)
)

for col in [
    "weighted_gap_asymmetry",
    "fragility_asymmetry"
]:

    s = df_1s[col]

    print(f"\n{col}")
    print(
        "Valid (%):",
        round(100 * s.notna().mean(), 4)
    )

    print(
        "Mean:",
        round(s.mean(), 6)
    )

    print(
        "Std:",
        round(s.std(), 6)
    )

    print(
        "Median:",
        round(s.median(), 6)
    )

    print(
        "P01:",
        round(s.quantile(0.01), 6)
    )

    print(
        "P99:",
        round(s.quantile(0.99), 6)
    )

print("\n===== COMPLETE =====")
print("Feature #35: weighted_gap_asymmetry")
print("Feature #36: fragility_asymmetry")
print("All existing features preserved.")


===== GAP WEIGHTS =====
Level 1->2: 0.536009
Level 2->3: 0.266174
Level 3->4: 0.132178
Level 4->5: 0.065638

===== FEATURE #35 AND #36 VALIDATION =====
Total rows: 524325
Valid books: 524325
Valid books (%): 100.0

weighted_gap_asymmetry
Valid (%): 100.0
Mean: -0.009745
Std: 0.414639
Median: 0.000956
P01: -1.159353
P99: 0.980433

fragility_asymmetry
Valid (%): 100.0
Mean: -0.00136
Std: 0.095268
Median: -0.002562
P01: -0.276629
P99: 0.286355

===== COMPLETE =====
Feature #35: weighted_gap_asymmetry
Feature #36: fragility_asymmetry
All existing features preserved.


In [69]:

import numpy as np
import pandas as pd

# ============================================================
# FEATURE #37 — RELATIVE LIQUIDITY MIGRATION (60 SECONDS)
#
# Exact mathematical definition from Experiment 31A.
#
# Output:
#   df_1s["relative_migration_60"]
#
# Requirements:
#   symbol, ts_sec
#   bid_p1...bid_p5
#   ask_p1...ask_p5
#   bid_q1...bid_q5
#   ask_q1...ask_q5
#
# Causal feature: current snapshot and t-60s only.
# No future information. No changes to existing columns.
# ============================================================

LOOKBACK = 60
EPS = 1e-12

BID_PRICE = [f"bid_p{i}" for i in range(1, 6)]
ASK_PRICE = [f"ask_p{i}" for i in range(1, 6)]
BID_QTY = [f"bid_q{i}" for i in range(1, 6)]
ASK_QTY = [f"ask_q{i}" for i in range(1, 6)]

REQUIRED = (
    ["symbol", "ts_sec"]
    + BID_PRICE + ASK_PRICE
    + BID_QTY + ASK_QTY
)

missing = sorted(set(REQUIRED) - set(df_1s.columns))

if missing:
    raise ValueError(f"Missing columns: {missing}")

# ------------------------------------------------------------
# 1. Extract source data; preserve original row identity
# ------------------------------------------------------------

work = df_1s[REQUIRED].copy()

work["_row_id"] = np.arange(len(work))

work["symbol"] = work["symbol"].astype(str)

for col in REQUIRED:
    if col != "symbol":
        work[col] = pd.to_numeric(
            work[col], errors="coerce"
        ).replace([np.inf, -np.inf], np.nan)

if work["ts_sec"].isna().any():
    raise ValueError("Invalid timestamps")

ts = work["ts_sec"].to_numpy(dtype=float)

if not np.all(ts == np.floor(ts)):
    raise ValueError("ts_sec must contain whole seconds")

work["ts_sec"] = work["ts_sec"].astype("int64")

if work.duplicated(["symbol", "ts_sec"]).any():
    raise ValueError("Duplicate symbol/timestamp keys")

# ------------------------------------------------------------
# 2. Determine IST session dates
# ------------------------------------------------------------

ist = pd.to_datetime(
    work["ts_sec"],
    unit="s",
    utc=True
).dt.tz_convert("Asia/Kolkata")

work["_session_date"] = ist.dt.date

# ------------------------------------------------------------
# 3. Read the five-level order book
# ------------------------------------------------------------

BP = work[BID_PRICE].to_numpy(dtype=np.float64)
AP = work[ASK_PRICE].to_numpy(dtype=np.float64)

BQ = work[BID_QTY].to_numpy(dtype=np.float64)
AQ = work[ASK_QTY].to_numpy(dtype=np.float64)

mid = (BP[:, 0] + AP[:, 0]) / 2.0

# ------------------------------------------------------------
# 4. Validate snapshots
# ------------------------------------------------------------

valid_book = (
    np.isfinite(BP).all(axis=1)
    & np.isfinite(AP).all(axis=1)
    & np.isfinite(BQ).all(axis=1)
    & np.isfinite(AQ).all(axis=1)
    & (BP > 0).all(axis=1)
    & (AP > 0).all(axis=1)
    & (AP[:, 0] >= BP[:, 0])
    & (np.diff(BP, axis=1) < 0).all(axis=1)
    & (np.diff(AP, axis=1) > 0).all(axis=1)
    & (BQ >= 0).all(axis=1)
    & (AQ >= 0).all(axis=1)
    & (BQ.sum(axis=1) > 0)
    & (AQ.sum(axis=1) > 0)
)

# ------------------------------------------------------------
# 5. Normalized five-level depth
# ------------------------------------------------------------

bid_total = BQ.sum(axis=1)
ask_total = AQ.sum(axis=1)

with np.errstate(divide="ignore", invalid="ignore"):

    pi_bid = BQ / bid_total[:, None]
    pi_ask = AQ / ask_total[:, None]

    bid_distance = (
        1e4 * (BP[:, [0]] - BP)
        / mid[:, None]
    )

    ask_distance = (
        1e4 * (AP - AP[:, [0]])
        / mid[:, None]
    )

# ------------------------------------------------------------
# 6. Liquidity center of mass
# ------------------------------------------------------------

mu_bid = np.sum(
    pi_bid * bid_distance,
    axis=1
)

mu_ask = np.sum(
    pi_ask * ask_distance,
    axis=1
)

mu_bid[~valid_book] = np.nan
mu_ask[~valid_book] = np.nan

mu_bid[~np.isfinite(mu_bid)] = np.nan
mu_ask[~np.isfinite(mu_ask)] = np.nan

work["_mu_bid"] = mu_bid
work["_mu_ask"] = mu_ask

# ------------------------------------------------------------
# 7. Exact t-60-second join
#
# No shift(60): missing seconds cannot distort
# the actual time horizon.
#
# The date key prevents crossing trading sessions.
# ------------------------------------------------------------

previous = work[
    ["symbol", "ts_sec",
     "_session_date", "_mu_bid", "_mu_ask"]
].copy()

previous["ts_sec"] += LOOKBACK

previous = previous.rename(
    columns={
        "_mu_bid": "_mu_bid_lag60",
        "_mu_ask": "_mu_ask_lag60"
    }
)

work = work.merge(
    previous,
    on=["symbol", "ts_sec", "_session_date"],
    how="left",
    validate="one_to_one",
    sort=False
)

# ------------------------------------------------------------
# 8. Changes in liquidity centers
# ------------------------------------------------------------

delta_bid = (
    work["_mu_bid"] - work["_mu_bid_lag60"]
)

delta_ask = (
    work["_mu_ask"] - work["_mu_ask_lag60"]
)

# ------------------------------------------------------------
# 9. Feature #37
# ------------------------------------------------------------

with np.errstate(divide="ignore", invalid="ignore"):

    numerator = delta_bid - delta_ask

    denominator = (
        delta_bid.abs()
        + delta_ask.abs()
        + EPS
    )

    relative_migration = (
        numerator / denominator
    )

relative_migration = relative_migration.replace(
    [np.inf, -np.inf], np.nan
)

work["relative_migration_60"] = (
    relative_migration
)

# ------------------------------------------------------------
# 10. Restore original row order
# ------------------------------------------------------------

work = work.sort_values("_row_id")

if len(work) != len(df_1s):
    raise ValueError("Unexpected change in row count")

df_1s["relative_migration_60"] = (
    work["relative_migration_60"].to_numpy(
        dtype=np.float64
    )
)

# ------------------------------------------------------------
# 11. Validation
# ------------------------------------------------------------

s = df_1s["relative_migration_60"]

print("===== FEATURE #37 VALIDATION =====")

print("Total rows:", len(df_1s))

print(
    "Valid (%):",
    round(100*s.notna().mean(), 4)
)

print(
    "Zero (%):",
    round(100*s.eq(0).mean(), 4)
)

print(
    "Minimum:",
    round(s.min(), 6)
)

print(
    "Maximum:",
    round(s.max(), 6)
)

print(
    "Mean:",
    round(s.mean(), 6)
)

print(
    "Standard deviation:",
    round(s.std(), 6)
)

print(
    "Median:",
    round(s.median(), 6)
)

print(
    "P01:",
    round(s.quantile(0.01), 6)
)

print(
    "P99:",
    round(s.quantile(0.99), 6)
)

if s.notna().any():
    assert s.dropna().between(
        -1.00000001, 1.00000001
    ).all(), "Feature outside [-1, 1]"

print("\nFeature #37 created successfully.")
print("All previous columns preserved.")


===== FEATURE #37 VALIDATION =====
Total rows: 524325
Valid (%): 99.7826
Zero (%): 3.631
Minimum: -1.0
Maximum: 1.0
Mean: 0.006916
Standard deviation: 0.832546
Median: 0.0
P01: -1.0
P99: 1.0

Feature #37 created successfully.
All previous columns preserved.


In [ ]:

import numpy as np
import pandas as pd
import json
import hashlib
from pathlib import Path

# ============================================================
# USDINR FINAL FEATURE FREEZE — RESEARCH V2
#
# REQUIRED:
#   core_31
#   usdinr_rebuilt_37_research
#   calendar_exp34b_features
#   calendar_exp36a_features
#   calendar_exp39a_features
#
# OUTPUTS:
#   usdinr_baseline_v2_46inputs.parquet
#   usdinr_challenger_v2_50inputs.parquet
#   usdinr_feature_manifest_v2.json
#
# Only original 16 research sessions (<= Oct 19).
# Do not include October 21–23 holdout.
#
# This is a registry/data-freeze step, not a
# fresh end-to-end computation from raw L5 data.
# ============================================================

OUTDIR = Path("usdinr_final_v2")
OUTDIR.mkdir(parents=True, exist_ok=True)

KEYS = ["symbol", "ts_sec"]

CORE31 = [
    "ofi_l1_sum_60s_signed_log",
    "ofi_rate_diff_30_300_sl",
    "ofi_activity_60s",
    "ofi_activity_300s",
    "update_rate_300s",
    "price_move_rate_30s",
    "impact_lambda_1800s",
    "impact_r2_1800s",
    "ofi_response_gap_z",
    "queue_imbalance_l1",
    "depth_imbalance_l5",
    "queue_replenishment_pressure_60s",
    "queue_imbalance_delta",
    "depth_imbalance_delta",
    "spread_rank_1800s",
    "price_mom_10s_new",
    "price_mom_30s_new",
    "price_mom_60s_new",
    "log_rv_300s_new",
    "rv_ratio_60_300_new",
    "ret_acf_lag1_300s",
    "log_vr_5_300s",
    "log_vr_30_300s",
    "semivar_imbalance_300s",
    "semivar_imbalance_900s",
    "sign_persistence_900s",
    "jump_ratio_300s",
    "range_position_300s",
    "spread_depth_stress_900",
    "recovery_available_300",
    "ofi_disagreement_weighted_60"
]

REBUILT6 = [
    "quote_retreat_imbalance_30_24a",
    "slope_asymmetry_28a",
    "curvature_asymmetry_28a",
    "weighted_gap_asymmetry_29a",
    "fragility_asymmetry_29a",
    "relative_migration_60_31a"
]

CONTROLS9 = [
    "queue_imbalance_28a",
    "depth_imbalance_28a",
    "log_total_depth_28a",
    "spread_bps_28a",
    "bid_l1_concentration_28a",
    "ask_l1_concentration_28a",
    "momentum_30_28a",
    "momentum_60_28a",
    "clock_seconds"
]

CHALLENGERS4 = [
    "recovery_pressure_34b",
    "ask_shock_age_34b",
    "side_update_imbalance_60_36a",
    "migration_activity_asym_60_39a"
]

BASE_INPUTS = CORE31 + REBUILT6 + CONTROLS9
CHALLENGER_INPUTS = BASE_INPUTS + CHALLENGERS4

assert len(CORE31) == 31
assert len(REBUILT6) == 6
assert len(CONTROLS9) == 9
assert len(CHALLENGERS4) == 4

assert len(BASE_INPUTS) == 46
assert len(CHALLENGER_INPUTS) == 50
assert len(set(CHALLENGER_INPUTS)) == 50

# ------------------------------------------------------------
# 1. Source availability checks
# ------------------------------------------------------------

REQUIRED_FRAMES = [
    "core_31",
    "usdinr_rebuilt_37_research",
    "calendar_exp34b_features",
    "calendar_exp36a_features",
    "calendar_exp39a_features"
]

missing_frames = [
    name for name in REQUIRED_FRAMES
    if name not in globals()
]

if missing_frames:
    raise ValueError(
        f"Missing frames: {missing_frames}. "
        "Load saved outputs or rerun their "
        "feature-construction cells."
    )

SOURCES = {
    "original31": core_31,
    "rebuilt": usdinr_rebuilt_37_research,
    "recovery": calendar_exp34b_features,
    "events": calendar_exp36a_features,
    "migration": calendar_exp39a_features
}

SOURCE_GROUPS = {
    "original31": CORE31,
    "rebuilt": REBUILT6 + CONTROLS9,
    "recovery": [
        "recovery_pressure_34b",
        "ask_shock_age_34b"
    ],
    "events": [
        "side_update_imbalance_60_36a"
    ],
    "migration": [
        "migration_activity_asym_60_39a"
    ]
}

# ------------------------------------------------------------
# 2. Freeze the exact research-key universe
#
# IMPORTANT: keys come from the already restricted
# research frame, not the full raw data.
# ------------------------------------------------------------

base_source = SOURCES["rebuilt"]

needed = KEYS + REBUILT6 + CONTROLS9

missing = [
    c for c in needed
    if c not in base_source.columns
]

if missing:
    raise ValueError(
        "Rebuilt frame missing: "
        + str(missing)
    )

research = base_source[KEYS].copy()

research["symbol"] = research[
    "symbol"
].astype(str)

research["ts_sec"] = pd.to_numeric(
    research["ts_sec"],
    errors="raise"
).astype(np.int64)

research["date"] = pd.to_datetime(
    research["ts_sec"],
    unit="s",
    utc=True
).dt.tz_convert(
    "Asia/Kolkata"
).dt.date

research = (
    research.loc[
        (research["date"].astype(str) <= "2026-10-19")
        & (research["date"].astype(str) != "2026-09-28")
    ]
    .sort_values(["date", "symbol", "ts_sec"])
    .reset_index(drop=True)
)

assert len(research) == 459122, (
    f"Expected 459122 research rows, "
    f"found {len(research)}"
)

assert not research.duplicated(KEYS).any()

sessions = research.groupby(
    ["symbol", "date"]
).ngroups

assert sessions == 16

# ------------------------------------------------------------
# 3. Exact keyed feature alignment
# ------------------------------------------------------------

def make_index(frame):
    return pd.MultiIndex.from_arrays(
        [
            frame["symbol"].astype(str),
            pd.to_numeric(
                frame["ts_sec"],
                errors="raise"
            ).astype(np.int64)
        ],
        names=KEYS
    )

target_index = make_index(research)

def align_source(frame, columns, label):

    missing = [
        c for c in KEYS + columns
        if c not in frame.columns
    ]

    if missing:
        raise ValueError(
            f"{label} missing columns: {missing}"
        )

    source_index = make_index(frame)

    if source_index.has_duplicates:
        raise ValueError(
            f"{label} has duplicate keys"
        )

    positions = source_index.get_indexer(
        target_index
    )

    if not (positions >= 0).all():
        raise ValueError(
            f"{label}: "
            f"{int((positions < 0).sum())} "
            "research keys missing"
        )

    result = {}

    for col in columns:
        values = pd.to_numeric(
            frame[col],
            errors="raise"
        ).to_numpy(dtype=np.float32)

        aligned = values[positions].copy()

        aligned[
            ~np.isfinite(aligned)
        ] = np.nan

        result[col] = aligned

    return pd.DataFrame(result)

feature_parts = {}

for source_name, columns in SOURCE_GROUPS.items():

    feature_parts[source_name] = align_source(
        SOURCES[source_name],
        columns,
        source_name
    )

# ------------------------------------------------------------
# 4. Construct frozen datasets
# ------------------------------------------------------------

features = pd.concat(
    [
        feature_parts["original31"],
        feature_parts["rebuilt"],
        feature_parts["recovery"],
        feature_parts["events"],
        feature_parts["migration"]
    ],
    axis=1
)

assert list(features.columns) == (
    CORE31
    + REBUILT6
    + CONTROLS9
    + CHALLENGERS4
)

frozen = pd.concat(
    [
        research[["symbol", "ts_sec", "date"]],
        features
    ],
    axis=1
)

# Preserve original historical research columns,
# including NaNs. Never substitute zeros for
# ask_shock_age when no shock was observed.

baseline = frozen[
    ["symbol", "ts_sec", "date"] + BASE_INPUTS
].copy()

challenger = frozen[
    ["symbol", "ts_sec", "date"] + CHALLENGER_INPUTS
].copy()

assert len(baseline) == len(challenger) == 459122

assert len(baseline.columns) == 49
assert len(challenger.columns) == 53

# ------------------------------------------------------------
# 5. Core data-quality checks
# ------------------------------------------------------------

# Compare frozen original 31 with source values.
assert np.array_equal(
    baseline[CORE31].isna().to_numpy(),
    feature_parts["original31"][CORE31].isna().to_numpy()
)

for col in CORE31:
    a = baseline[col].to_numpy()
    b = feature_parts["original31"][col].to_numpy()

    if not np.array_equal(
        a, b, equal_nan=True
    ):
        raise ValueError(
            f"Original 31 preservation failed: {col}"
        )

# Dataset contains historical observations only.
assert str(baseline["date"].max()) <= "2026-10-19"

# No target columns in input schema.
assert not any(
    c.startswith(("future_", "target_", "label_"))
    for c in CHALLENGER_INPUTS
)

# Both datasets have exactly matching keys.
assert baseline[KEYS].equals(
    challenger[KEYS]
)

# Validate known bounds.
BOUNDED = {
    "relative_migration_60_31a": (-1, 1),
    "queue_imbalance_28a": (-1, 1),
    "depth_imbalance_28a": (-1, 1),
    "side_update_imbalance_60_36a": (-1, 1)
}

for col, (lo, hi) in BOUNDED.items():

    v = challenger[col].dropna()

    if not v.between(
        lo - 1e-5,
        hi + 1e-5
    ).all():
        raise ValueError(
            f"Bounds check failed: {col}"
        )

age = challenger[
    "ask_shock_age_34b"
].dropna()

assert age.between(0, 60).all()

# ------------------------------------------------------------
# 6. Model sample diagnostics
# ------------------------------------------------------------

base_complete = np.isfinite(
    baseline[
        BASE_INPUTS
    ].to_numpy(dtype=np.float32)
).all(axis=1)

print("===== FINAL FEATURE FREEZE =====")
print("Research rows:", len(baseline))
print("Research sessions:", sessions)
print("First date:", research["date"].min())
print("Last date:", research["date"].max())
print("Original features:", len(CORE31))
print("Reconstructed features:", len(REBUILT6))
print("Controls:", len(CONTROLS9))
print("Challengers:", len(CHALLENGERS4))

print("\nBaseline inputs:", len(BASE_INPUTS))
print("Challenger inputs:", len(CHALLENGER_INPUTS))

print("\n===== COVERAGE =====")

coverage_rows = []

for name in CHALLENGER_INPUTS:

    s = challenger[name]

    coverage_rows.append({
        "feature": name,
        "coverage_pct": 100 * s.notna().mean(),
        "missing_pct": 100 * s.isna().mean(),
        "std": s.std()
    })

coverage_table = pd.DataFrame(
    coverage_rows
)

print(
    coverage_table.loc[
        coverage_table["feature"].isin(
            CHALLENGERS4
        )
    ]
    .round(5)
    .to_string(index=False)
)

print(
    "\nBaseline-complete rows:",
    int(base_complete.sum())
)

print(
    "Baseline-complete coverage:",
    round(
        100 * base_complete.mean(),
        5
    ), "%"
)

# ------------------------------------------------------------
# 7. Provenance and model schema
# ------------------------------------------------------------

def schema_digest(columns):
    serialized = json.dumps(
        columns,
        separators=(",", ":")
    )
    return hashlib.sha256(
        serialized.encode("utf-8")
    ).hexdigest()

manifest = {
    "version": "research_v2",
    "instrument": "USDINR futures",
    "time_resolution": "1_second",
    "session_timezone": "Asia/Kolkata",
    "research_end_date": "2026-10-19",
    "holdout_dates_reserved": [
        "2026-10-21",
        "2026-10-22",
        "2026-10-23"
    ],
    "research_rows": int(len(research)),
    "research_sessions": int(sessions),

    "baseline_engineered_count": 37,
    "baseline_input_count": 46,
    "challenger_engineered_count": 41,
    "challenger_input_count": 50,

    "original31": CORE31,
    "reconstructed6": REBUILT6,
    "controls9": CONTROLS9,
    "challengers4": CHALLENGERS4,

    "baseline_columns": BASE_INPUTS,
    "challenger_columns": CHALLENGER_INPUTS,

    "baseline_schema_sha256":
        schema_digest(BASE_INPUTS),

    "challenger_schema_sha256":
        schema_digest(CHALLENGER_INPUTS),

    "baseline_complete_rows":
        int(base_complete.sum()),

    "baseline_complete_coverage_pct":
        float(100 * base_complete.mean()),

    "formula_provenance": {
        "original31":
            "Values from original 31-feature parquet; "
            "underlying source implementations must "
            "be retained for future raw-data rebuilds.",

        "reconstructed6":
            "Research V2 reconstructed definitions. "
            "Exact original implementation parity "
            "is not established for all features.",

        "controls9":
            "Research V2 reconstructed controls; "
            "historical formula parity unverified.",

        "challengers4":
            "Calculated using historical experiment "
            "34B, 36A and 39A feature constructors."
    },

    "missing_value_policy": {
        "model": "LightGBM native NaN support",
        "ask_shock_age_34b":
            "NaN means no active qualifying ask shock",
        "no_global_imputation": True
    },

    "production_status":
        "RESEARCH FREEZE — NOT YET LIVE READY",

    "limitations": [
        "Some historical feature definitions "
        "were reconstructed.",
        "Holdout features must be generated "
        "with frozen causal implementations.",
        "No transaction cost or fill simulation "
        "has been completed.",
        "No independent holdout performance "
        "has been measured."
    ]
}

# ------------------------------------------------------------
# 8. Save frozen artifacts
# ------------------------------------------------------------

baseline_path = (
    OUTDIR / "usdinr_baseline_v2_46inputs.parquet"
)

challenger_path = (
    OUTDIR / "usdinr_challenger_v2_50inputs.parquet"
)

manifest_path = (
    OUTDIR / "usdinr_feature_manifest_v2.json"
)

baseline.to_parquet(
    baseline_path,
    index=False
)

challenger.to_parquet(
    challenger_path,
    index=False
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        manifest,
        f,
        indent=2
    )

# Reload and verify saved schemas.
check_base = pd.read_parquet(
    baseline_path
)

check_challenger = pd.read_parquet(
    challenger_path
)

assert list(check_base.columns) == (
    ["symbol", "ts_sec", "date"] + BASE_INPUTS
)

assert list(check_challenger.columns) == (
    ["symbol", "ts_sec", "date"] + CHALLENGER_INPUTS
)

assert len(check_base) == len(check_challenger)

del check_base, check_challenger

print("\n===== SAVED FILES =====")

for path in [
    baseline_path,
    challenger_path,
    manifest_path
]:
    print(
        str(path.resolve()),
        "bytes=", path.stat().st_size
    )

print("\n===== FREEZE COMPLETE =====")
print("Baseline: 37 engineered + 9 controls")
print("Challenger: 41 engineered + 9 controls")
print("Original 31 preserved")
print("Target columns excluded")
print("Historical research dates only")
print("Holdout untouched")
print("Ready for versioned model development")
